In [ ]:
# This code checks one MTSD annotation JSON file to understand its structure.
# It prints the main keys and a small sample of the annotation information.
# It only reads the file and does not change or delete any existing data.

import os
import json

# Set the folder where the original MTSD annotation JSON files are stored.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Get a list of JSON files in the annotation folder.
json_file_names = []

for file_name in os.listdir(annotation_folder):
    if file_name.endswith(".json"):
        json_file_names.append(file_name)

# Check whether at least one JSON file exists.
if len(json_file_names) == 0:
    print("No JSON annotation files were found.")
else:
    # Select one JSON file for inspection.
    selected_file_name = json_file_names[0]
    selected_file_path = os.path.join(annotation_folder, selected_file_name)

    print("Selected annotation file:")
    print(selected_file_path)
    print()

    # Open the JSON file and load its contents.
    with open(selected_file_path, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)

    # Print the top-level keys so we can understand the data structure.
    print("Top-level keys:")
    print(list(annotation_data.keys()))
    print()

    # Print the annotation information.
    # The output is limited to avoid printing a very large JSON file.
    print("Annotation data preview:")
    print(json.dumps(annotation_data, indent=2, ensure_ascii=False)[:6000])

In [ ]:
# This code checks all MTSD annotation JSON files and summarizes the dataset.
# It counts annotated objects, object categories, and several annotation properties.
# It also checks whether any bounding boxes have invalid coordinates.
# The code only reads the annotation files and does not modify the dataset.

import os
import json
from collections import Counter

# Set the path to the folder containing the original MTSD annotation files.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Create counters to store the annotation statistics.
total_json_files = 0
total_objects = 0
total_objects_included = 0
total_objects_not_included = 0
total_objects_ambiguous = 0
total_objects_occluded = 0
total_objects_out_of_frame = 0
total_invalid_bounding_boxes = 0

# Store the number of objects belonging to each category.
category_counter = Counter()

# Store the number of images that contain at least one object.
images_with_objects = 0

# Store the number of JSON files that cannot be read successfully.
unreadable_json_files = []

# Get all JSON filenames in the annotation folder.
json_file_names = []

for file_name in os.listdir(annotation_folder):
    if file_name.endswith(".json"):
        json_file_names.append(file_name)

print("Starting to check the MTSD annotation dataset...")
print("Number of JSON files found:", len(json_file_names))
print()

# Read every JSON annotation file one by one.
for file_name in json_file_names:

    # Build the complete path of the current annotation file.
    file_path = os.path.join(annotation_folder, file_name)

    try:
        # Open the JSON file and load its annotation information.
        with open(file_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        total_json_files = total_json_files + 1

        # Get the list of annotated objects from the current image.
        objects = annotation_data.get("objects", [])

        # Count images that contain at least one annotated object.
        if len(objects) > 0:
            images_with_objects = images_with_objects + 1

        # Process each annotated object in the current image.
        for object_data in objects:

            total_objects = total_objects + 1

            # Read the object category.
            category_name = object_data.get("label", "unknown")
            category_counter[category_name] = category_counter[category_name] + 1

            # Read the additional properties of the object.
            properties = object_data.get("properties", {})

            # Count whether the object is included in the annotation.
            if properties.get("included", True):
                total_objects_included = total_objects_included + 1
            else:
                total_objects_not_included = total_objects_not_included + 1

            # Count ambiguous objects.
            if properties.get("ambiguous", False):
                total_objects_ambiguous = total_objects_ambiguous + 1

            # Count occluded objects.
            if properties.get("occluded", False):
                total_objects_occluded = total_objects_occluded + 1

            # Count objects that extend outside the image frame.
            if properties.get("out-of-frame", False):
                total_objects_out_of_frame = total_objects_out_of_frame + 1

            # Check whether the bounding box exists and has valid coordinates.
            bounding_box = object_data.get("bbox", {})

            if (
                "xmin" not in bounding_box
                or "ymin" not in bounding_box
                or "xmax" not in bounding_box
                or "ymax" not in bounding_box
            ):
                total_invalid_bounding_boxes = total_invalid_bounding_boxes + 1

            else:
                xmin = bounding_box["xmin"]
                ymin = bounding_box["ymin"]
                xmax = bounding_box["xmax"]
                ymax = bounding_box["ymax"]

                image_width = annotation_data.get("width", 0)
                image_height = annotation_data.get("height", 0)

                # A valid bounding box must have positive width and height.
                # Its coordinates should also stay inside the image boundaries.
                if (
                    xmin >= xmax
                    or ymin >= ymax
                    or xmin < 0
                    or ymin < 0
                    or xmax > image_width
                    or ymax > image_height
                ):
                    total_invalid_bounding_boxes = total_invalid_bounding_boxes + 1

    except Exception as error:
        # Record the filename if the annotation file cannot be read.
        unreadable_json_files.append((file_name, str(error)))

# Print the overall annotation statistics.
print("========== MTSD Annotation Summary ==========")
print("Successfully read JSON files:", total_json_files)
print("Images containing at least one object:", images_with_objects)
print("Total annotated objects:", total_objects)
print("Objects with included = True:", total_objects_included)
print("Objects with included = False:", total_objects_not_included)
print("Objects marked as ambiguous:", total_objects_ambiguous)
print("Objects marked as occluded:", total_objects_occluded)
print("Objects marked as out-of-frame:", total_objects_out_of_frame)
print("Invalid bounding boxes:", total_invalid_bounding_boxes)
print("Unreadable JSON files:", len(unreadable_json_files))
print()

# Print the most common object categories.
print("========== Top 30 Object Categories ==========")

for category_name, category_count in category_counter.most_common(30):
    print(category_name, ":", category_count)

# Print any annotation files that could not be read.
if len(unreadable_json_files) > 0:
    print()
    print("========== Unreadable Files ==========")

    for file_name, error_message in unreadable_json_files[:10]:
        print(file_name, ":", error_message)

In [ ]:
# This code counts object categories separately for included and excluded annotations.
# It also checks how many included objects belong to the other-sign category.
# The purpose is to help us decide which objects should be used to train YOLO.
# This code only reads JSON files and does not modify any existing data.

import os
import json
from collections import Counter

# Set the path to the original MTSD annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Create separate counters for included and excluded objects.
included_category_counter = Counter()
excluded_category_counter = Counter()

# Count the total number of objects in each group.
included_object_count = 0
excluded_object_count = 0

# Count how many included objects are other-sign.
included_other_sign_count = 0

# Read every JSON annotation file.
for file_name in os.listdir(annotation_folder):

    # Process only JSON files.
    if file_name.endswith(".json"):

        file_path = os.path.join(annotation_folder, file_name)

        # Open and read the annotation file.
        with open(file_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        # Get all annotated objects in this image.
        objects = annotation_data.get("objects", [])

        # Count each object according to its included property.
        for object_data in objects:

            label = object_data.get("label", "unknown")
            properties = object_data.get("properties", {})

            # Put the object into the included or excluded group.
            if properties.get("included", True) is True:

                included_category_counter[label] += 1
                included_object_count += 1

                if label == "other-sign":
                    included_other_sign_count += 1

            else:

                excluded_category_counter[label] += 1
                excluded_object_count += 1

# Print the statistics for included objects.
print("========== Included Objects ==========")
print("Total included objects:", included_object_count)
print("Included other-sign objects:", included_other_sign_count)
print("Number of included categories:", len(included_category_counter))
print()

print("Top 30 included categories:")

for category_name, category_count in included_category_counter.most_common(30):
    print(category_name, ":", category_count)

# Print the statistics for excluded objects.
print()
print("========== Excluded Objects ==========")
print("Total excluded objects:", excluded_object_count)
print("Number of excluded categories:", len(excluded_category_counter))
print()

print("Top 15 excluded categories:")

for category_name, category_count in excluded_category_counter.most_common(15):
    print(category_name, ":", category_count)

In [ ]:
# This code compares the categories in the MTSD annotations with the 326 classes
# used by the two existing CNN classification models.
# It checks which included categories match the CNN class mapping and which do not.
# It also counts how many included objects belong to categories outside the CNN mapping.
# This code only reads existing files and does not modify the dataset.

import os
import json
from collections import Counter

# Set the path to the original annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the path to the class mapping file used by the existing CNN models.
class_mapping_path = r".\train\final_class_to_index.json"

# Load the class mapping created for the CNN classification dataset.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

# Create a set of class names for easier category comparison.
cnn_class_names = set(class_to_index.keys())

# Create counters for all included annotation categories and their object counts.
included_category_counter = Counter()

# Read every JSON annotation file.
for file_name in os.listdir(annotation_folder):

    # Only process files ending with .json.
    if file_name.endswith(".json"):

        # Create the complete path to the annotation file.
        file_path = os.path.join(annotation_folder, file_name)

        # Load the annotation information.
        with open(file_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        # Get the list of objects in the current image.
        objects = annotation_data.get("objects", [])

        # Check every annotated object.
        for object_data in objects:

            # Read the annotation properties and category name.
            properties = object_data.get("properties", {})
            category_name = object_data.get("label", "unknown")

            # Count categories only when included is True.
            if properties.get("included", True) is True:
                included_category_counter[category_name] += 1

# Find included categories that also exist in the CNN class mapping.
matching_categories = []
for category_name in included_category_counter:
    if category_name in cnn_class_names:
        matching_categories.append(category_name)

# Find included categories that do not exist in the CNN class mapping.
unmatched_categories = []
for category_name in included_category_counter:
    if category_name not in cnn_class_names:
        unmatched_categories.append(category_name)

# Count objects belonging to matching and unmatched categories.
matching_object_count = 0
for category_name in matching_categories:
    matching_object_count += included_category_counter[category_name]

unmatched_object_count = 0
for category_name in unmatched_categories:
    unmatched_object_count += included_category_counter[category_name]

# Print the comparison results.
print("========== CNN and Annotation Category Comparison ==========")
print("Number of CNN classes:", len(cnn_class_names))
print("Number of included annotation categories:", len(included_category_counter))
print("Included categories matching CNN classes:", len(matching_categories))
print("Included categories not matching CNN classes:", len(unmatched_categories))
print()

print("Included objects in matching categories:", matching_object_count)
print("Included objects in unmatched categories:", unmatched_object_count)
print()

# Print the categories that do not match the CNN class mapping.
print("========== Unmatched Included Categories ==========")

for category_name in sorted(unmatched_categories):
    print(category_name, ":", included_category_counter[category_name])

# Print the most common matching categories.
print()
print("========== Top 20 Matching Included Categories ==========")

matching_category_counter = Counter()

for category_name in matching_categories:
    matching_category_counter[category_name] = included_category_counter[category_name]

for category_name, object_count in matching_category_counter.most_common(20):
    print(category_name, ":", object_count)

In [ ]:
# This code counts all annotation objects that match the 326 existing CNN classes.
# It separates included and excluded objects and counts the remaining unmatched objects.
# The results will help us choose which bounding boxes can be used to train YOLO.
# This code only reads existing files and does not change the original dataset.

import os
import json
from collections import Counter

# Set the folder containing the original MTSD JSON annotations.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the path to the class mapping used by the existing CNN models.
class_mapping_path = r".\train\final_class_to_index.json"

# Load the existing CNN class mapping.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

# Create a set containing all supported CNN class names.
cnn_class_names = set(class_to_index.keys())

# Create counters for matching categories.
included_matching_counter = Counter()
excluded_matching_counter = Counter()

# Count objects that do not match the CNN class mapping.
included_unmatched_counter = Counter()
excluded_unmatched_counter = Counter()

# Count all objects processed.
total_objects_checked = 0

# Read all annotation JSON files.
for file_name in os.listdir(annotation_folder):

    if file_name.endswith(".json"):

        file_path = os.path.join(annotation_folder, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        objects = annotation_data.get("objects", [])

        # Process each object separately.
        for object_data in objects:

            total_objects_checked += 1

            category_name = object_data.get("label", "unknown")
            properties = object_data.get("properties", {})

            # Read the included property.
            is_included = properties.get("included", True)

            # Count categories that match the existing CNN classes.
            if category_name in cnn_class_names:

                if is_included:
                    included_matching_counter[category_name] += 1
                else:
                    excluded_matching_counter[category_name] += 1

            # Count categories that do not match the existing CNN classes.
            else:

                if is_included:
                    included_unmatched_counter[category_name] += 1
                else:
                    excluded_unmatched_counter[category_name] += 1

# Calculate the number of matching objects in each group.
included_matching_total = sum(included_matching_counter.values())
excluded_matching_total = sum(excluded_matching_counter.values())

included_unmatched_total = sum(included_unmatched_counter.values())
excluded_unmatched_total = sum(excluded_unmatched_counter.values())

# Print the overall counts.
print("========== Full Annotation and CNN Class Comparison ==========")
print("Total annotation objects checked:", total_objects_checked)
print("CNN classes:", len(cnn_class_names))
print()

print("Included objects matching CNN classes:", included_matching_total)
print("Excluded objects matching CNN classes:", excluded_matching_total)
print("Included objects not matching CNN classes:", included_unmatched_total)
print("Excluded objects not matching CNN classes:", excluded_unmatched_total)
print()

# Print the most common matching categories in the excluded group.
print("========== Top 20 Excluded Objects Matching CNN Classes ==========")

for category_name, object_count in excluded_matching_counter.most_common(20):
    print(category_name, ":", object_count)

# Print the most common unmatched categories in the included group.
print()
print("========== Top 15 Included Objects Not Matching CNN Classes ==========")

for category_name, object_count in included_unmatched_counter.most_common(15):
    print(category_name, ":", object_count)

# Print the most common unmatched categories in the excluded group.
print()
print("========== Top 15 Excluded Objects Not Matching CNN Classes ==========")

for category_name, object_count in excluded_unmatched_counter.most_common(15):
    print(category_name, ":", object_count)

In [ ]:
# This code counts CNN-compatible traffic sign annotations in the official
# training and validation splits.
# It matches each annotation JSON filename with the scene IDs in train.txt and val.txt.
# It counts matching objects, included and excluded objects, and annotated scenes.
# It only reads existing files and does not modify any dataset files.

import os
import json
from collections import Counter

# Set the folder containing the original MTSD annotation JSON files.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the paths to the official dataset split files.
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"
val_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\val.txt"

# Set the path to the class mapping used by the existing CNN models.
class_mapping_path = r".\train\final_class_to_index.json"

# Load the 326 classes used by the existing CNN models.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

cnn_class_names = set(class_to_index.keys())

# Read the official training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            train_scene_ids.add(scene_id)

# Read the official validation scene IDs.
with open(val_split_path, "r", encoding="utf-8") as file:
    val_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            val_scene_ids.add(scene_id)

# Create counters for training annotations.
train_category_counter = Counter()
train_included_counter = Counter()
train_excluded_counter = Counter()

# Create counters for validation annotations.
val_category_counter = Counter()
val_included_counter = Counter()
val_excluded_counter = Counter()

# Track the number of scenes containing matching objects.
train_scenes_with_matching_objects = set()
val_scenes_with_matching_objects = set()

# Count how many annotation files belong to each official split.
train_json_count = 0
val_json_count = 0
unmatched_json_count = 0

# Get all annotation JSON filenames.
json_file_names = []

for file_name in os.listdir(annotation_folder):
    if file_name.endswith(".json"):
        json_file_names.append(file_name)

# Process each annotation JSON file.
for file_name in json_file_names:

    # The JSON filename without its extension is the scene ID.
    scene_id = os.path.splitext(file_name)[0]

    # Determine whether this annotation belongs to Train or Validation.
    if scene_id in train_scene_ids:
        split_name = "train"
        train_json_count += 1

    elif scene_id in val_scene_ids:
        split_name = "val"
        val_json_count += 1

    else:
        unmatched_json_count += 1
        continue

    # Build the complete path to the current annotation file.
    file_path = os.path.join(annotation_folder, file_name)

    # Load the annotation information.
    with open(file_path, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)

    # Get the objects annotated in this scene.
    objects = annotation_data.get("objects", [])

    # Process each object separately.
    for object_data in objects:

        # Read the category name and annotation properties.
        category_name = object_data.get("label", "unknown")
        properties = object_data.get("properties", {})

        # Count only categories supported by the existing CNN models.
        if category_name in cnn_class_names:

            if split_name == "train":
                train_category_counter[category_name] += 1

                if properties.get("included", True):
                    train_included_counter[category_name] += 1
                else:
                    train_excluded_counter[category_name] += 1

                train_scenes_with_matching_objects.add(scene_id)

            elif split_name == "val":
                val_category_counter[category_name] += 1

                if properties.get("included", True):
                    val_included_counter[category_name] += 1
                else:
                    val_excluded_counter[category_name] += 1

                val_scenes_with_matching_objects.add(scene_id)

# Calculate the total number of matching objects in each split.
train_total_matching = sum(train_category_counter.values())
val_total_matching = sum(val_category_counter.values())

train_total_included = sum(train_included_counter.values())
train_total_excluded = sum(train_excluded_counter.values())

val_total_included = sum(val_included_counter.values())
val_total_excluded = sum(val_excluded_counter.values())

# Print the official split matching results.
print("========== Official Split Matching ==========")
print("CNN classes:", len(cnn_class_names))
print("Train scene IDs:", len(train_scene_ids))
print("Validation scene IDs:", len(val_scene_ids))
print("Train annotation JSON files matched:", train_json_count)
print("Validation annotation JSON files matched:", val_json_count)
print("JSON files outside these two splits:", unmatched_json_count)
print()

# Print the training statistics.
print("========== Training Split ==========")
print("Matching objects:", train_total_matching)
print("Included matching objects:", train_total_included)
print("Excluded matching objects:", train_total_excluded)
print("Scenes with matching objects:", len(train_scenes_with_matching_objects))
print("Number of matching categories:", len(train_category_counter))
print()

# Print the validation statistics.
print("========== Validation Split ==========")
print("Matching objects:", val_total_matching)
print("Included matching objects:", val_total_included)
print("Excluded matching objects:", val_total_excluded)
print("Scenes with matching objects:", len(val_scenes_with_matching_objects))
print("Number of matching categories:", len(val_category_counter))
print()

# Print the most common matching categories in the training split.
print("========== Top 15 Training Categories ==========")

for category_name, object_count in train_category_counter.most_common(15):
    print(category_name, ":", object_count)

# Print the most common matching categories in the validation split.
print()
print("========== Top 15 Validation Categories ==========")

for category_name, object_count in val_category_counter.most_common(15):
    print(category_name, ":", object_count)

In [ ]:
# This code examines the annotation properties of objects whose categories
# match the 326 classes used by the existing CNN classification models.
# It separates training and validation statistics and counts ambiguous,
# occluded, and out-of-frame objects in both included and excluded groups.
# It only reads existing annotation files and does not modify the dataset.

import os
import json
from collections import defaultdict

# Set the original annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the official dataset split files.
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"
val_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\val.txt"

# Set the existing CNN class mapping.
class_mapping_path = r".\train\final_class_to_index.json"

# Load the CNN category names.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

cnn_class_names = set(class_to_index.keys())

# Read the training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            train_scene_ids.add(scene_id)

# Read the validation scene IDs.
with open(val_split_path, "r", encoding="utf-8") as file:
    val_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            val_scene_ids.add(scene_id)

# Create a dictionary to store the statistics for each split and group.
statistics = defaultdict(lambda: {
    "total": 0,
    "included": 0,
    "excluded": 0,
    "ambiguous": 0,
    "occluded": 0,
    "out_of_frame": 0,
    "included_ambiguous": 0,
    "excluded_ambiguous": 0,
    "included_occluded": 0,
    "excluded_occluded": 0,
    "included_out_of_frame": 0,
    "excluded_out_of_frame": 0
})

# Read every JSON annotation file.
for file_name in os.listdir(annotation_folder):

    if file_name.endswith(".json"):

        # Extract the scene ID from the JSON filename.
        scene_id = os.path.splitext(file_name)[0]

        # Determine whether the scene belongs to Train or Validation.
        if scene_id in train_scene_ids:
            split_name = "Train"

        elif scene_id in val_scene_ids:
            split_name = "Validation"

        else:
            continue

        # Open the current annotation file.
        file_path = os.path.join(annotation_folder, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        # Get all annotated objects in this scene.
        objects = annotation_data.get("objects", [])

        # Check each annotated object separately.
        for object_data in objects:

            category_name = object_data.get("label", "unknown")

            # Only analyze categories supported by the existing CNN models.
            if category_name not in cnn_class_names:
                continue

            properties = object_data.get("properties", {})

            is_included = properties.get("included", True)
            is_ambiguous = properties.get("ambiguous", False)
            is_occluded = properties.get("occluded", False)
            is_out_of_frame = properties.get("out-of-frame", False)

            # Select the statistics group for the current split.
            if is_included:
                group_name = "included"
            else:
                group_name = "excluded"

            key = (split_name, group_name)

            # Count the current object.
            statistics[key]["total"] += 1
            statistics[key][group_name] += 1

            # Count objects with ambiguous annotations.
            if is_ambiguous:
                statistics[key]["ambiguous"] += 1

                if is_included:
                    statistics[key]["included_ambiguous"] += 1
                else:
                    statistics[key]["excluded_ambiguous"] += 1

            # Count objects that are partially or fully occluded.
            if is_occluded:
                statistics[key]["occluded"] += 1

                if is_included:
                    statistics[key]["included_occluded"] += 1
                else:
                    statistics[key]["excluded_occluded"] += 1

            # Count objects that extend outside the image frame.
            if is_out_of_frame:
                statistics[key]["out_of_frame"] += 1

                if is_included:
                    statistics[key]["included_out_of_frame"] += 1
                else:
                    statistics[key]["excluded_out_of_frame"] += 1

# Print the statistics for each split and group.
for split_name in ["Train", "Validation"]:

    print()
    print("==========", split_name, "==========")

    for group_name in ["included", "excluded"]:

        key = (split_name, group_name)
        values = statistics[key]

        print()
        print("Group:", group_name)
        print("Total objects:", values["total"])
        print("Ambiguous objects:", values["ambiguous"])
        print("Occluded objects:", values["occluded"])
        print("Out-of-frame objects:", values["out_of_frame"])

        # Calculate percentages to make the annotation quality easier to compare.
        if values["total"] > 0:

            ambiguous_percentage = (
                values["ambiguous"] / values["total"]
            ) * 100

            occluded_percentage = (
                values["occluded"] / values["total"]
            ) * 100

            out_of_frame_percentage = (
                values["out_of_frame"] / values["total"]
            ) * 100

            print("Ambiguous percentage:", round(ambiguous_percentage, 2), "%")
            print("Occluded percentage:", round(occluded_percentage, 2), "%")
            print("Out-of-frame percentage:", round(out_of_frame_percentage, 2), "%")

In [ ]:
# This code finds a training image with multiple annotated objects.
# It draws the original MTSD bounding boxes and category names on the image.
# Different colors distinguish included objects from excluded objects.
# The visualization is saved as a new file for manual inspection.
# The original images and annotation files are not modified.

import os
import json
import random
from PIL import Image, ImageDraw, ImageFont

# Set the annotation folder and the official training split file.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"

# Set the folders containing the original road images.
image_folders = [
    r".\data\raw_images\train_part_1\images",
    r".\data\raw_images\train_part_2\images",
    r".\data\raw_images\train_part_3\images",
    r".\data\raw_images\val\images"
]

# Set the output path for the visualization.
output_folder = r".\Detection_Inspection"
os.makedirs(output_folder, exist_ok=True)

# Read the official training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            train_scene_ids.add(scene_id)

# Find images that contain several annotated objects.
candidate_scene_ids = []

for scene_id in train_scene_ids:

    annotation_path = os.path.join(annotation_folder, scene_id + ".json")

    if os.path.exists(annotation_path):

        with open(annotation_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        objects = annotation_data.get("objects", [])

        # Select scenes containing at least three annotated objects.
        if len(objects) >= 3:
            candidate_scene_ids.append(scene_id)

print("Training scenes with at least three annotated objects:", len(candidate_scene_ids))

# Randomly choose one scene to inspect.
if len(candidate_scene_ids) == 0:

    print("No suitable scene was found.")

else:

    random.seed(42)
    selected_scene_id = random.choice(candidate_scene_ids)

    print("Selected scene ID:", selected_scene_id)

    # Find the original image file in the available image folders.
    selected_image_path = None

    for image_folder in image_folders:

        possible_extensions = [".jpg", ".jpeg", ".png"]

        for extension in possible_extensions:

            possible_path = os.path.join(
                image_folder,
                selected_scene_id + extension
            )

            if os.path.exists(possible_path):
                selected_image_path = possible_path
                break

        if selected_image_path is not None:
            break

    # Load the selected annotation JSON.
    annotation_path = os.path.join(
        annotation_folder,
        selected_scene_id + ".json"
    )

    with open(annotation_path, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)

    # Check whether the original image was found.
    if selected_image_path is None:

        print("The original image could not be found in the listed folders.")

    else:

        # Open the original image.
        image = Image.open(selected_image_path).convert("RGB")

        # Create a drawing object for the image.
        draw = ImageDraw.Draw(image)

        # Try to load a readable font.
        try:
            font = ImageFont.truetype("arial.ttf", 22)
        except OSError:
            font = ImageFont.load_default()

        # Draw every annotated object, including objects outside the CNN classes.
        for object_data in annotation_data.get("objects", []):

            bounding_box = object_data.get("bbox", {})
            properties = object_data.get("properties", {})

            # Read the bounding box coordinates.
            xmin = bounding_box.get("xmin", 0)
            ymin = bounding_box.get("ymin", 0)
            xmax = bounding_box.get("xmax", 0)
            ymax = bounding_box.get("ymax", 0)

            # Skip invalid boxes that have no positive width or height.
            if xmax <= xmin or ymax <= ymin:
                continue

            # Use green for included objects and orange for excluded objects.
            if properties.get("included", True):
                box_color = "lime"
                status_text = "IN"
            else:
                box_color = "orange"
                status_text = "OUT"

            # Read the annotation category.
            category_name = object_data.get("label", "unknown")

            # Draw the bounding box on the original image.
            draw.rectangle(
                [xmin, ymin, xmax, ymax],
                outline=box_color,
                width=5
            )

            # Add the category name and inclusion status.
            label_text = status_text + ": " + category_name

            # Draw a filled background behind the text.
            text_box = draw.textbbox((xmin, ymin), label_text, font=font)

            draw.rectangle(
                text_box,
                fill=box_color
            )

            draw.text(
                (xmin, ymin),
                label_text,
                fill="black",
                font=font
            )

        # Save the inspection image to the new output folder.
        output_path = os.path.join(
            output_folder,
            selected_scene_id + "_annotation_inspection.jpg"
        )

        image.save(output_path)

        print("Original image:", selected_image_path)
        print("Inspection image saved to:", output_path)
        print("Number of annotated objects:", len(annotation_data.get("objects", [])))

In [ ]:
# This code prints the annotation details for one selected MTSD road image.
# It displays each object's category, bounding box coordinates, and properties.
# It helps us check whether the overlapping labels correspond to separate objects.
# This code only reads the JSON file and does not modify any existing data.

import os
import json

# Set the scene ID of the image we inspected.
scene_id = "4t7HLS11620L0KGg8K2FoA"

# Set the path to the original annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Build the complete path to the selected JSON file.
annotation_path = os.path.join(
    annotation_folder,
    scene_id + ".json"
)

# Open and load the annotation data.
with open(annotation_path, "r", encoding="utf-8") as file:
    annotation_data = json.load(file)

# Print the original image dimensions.
print("Image width:", annotation_data.get("width"))
print("Image height:", annotation_data.get("height"))
print()

# Print every object's details.
for object_number, object_data in enumerate(
    annotation_data.get("objects", []),
    start=1
):

    print("========== Object", object_number, "==========")

    print("Category:", object_data.get("label"))
    print("Bounding box:", object_data.get("bbox"))

    properties = object_data.get("properties", {})

    print("Included:", properties.get("included"))
    print("Ambiguous:", properties.get("ambiguous"))
    print("Occluded:", properties.get("occluded"))
    print("Out of frame:", properties.get("out-of-frame"))
    print("All properties:", properties)
    print()

In [ ]:
# This code counts valid bounding boxes for the existing 326 CNN classes.
# It separates included and excluded annotations for the official training
# and validation splits, and checks whether each bounding box fits inside
# the original image boundaries.
# The code only reads existing files and does not modify the dataset.

import os
import json
from collections import Counter

# Set the original annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the official training and validation split files.
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"
val_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\val.txt"

# Set the existing CNN class mapping.
class_mapping_path = r".\train\final_class_to_index.json"

# Load the class names supported by the existing CNN models.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

cnn_class_names = set(class_to_index.keys())

# Read the official training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id:
            train_scene_ids.add(scene_id)

# Read the official validation scene IDs.
with open(val_split_path, "r", encoding="utf-8") as file:
    val_scene_ids = set()

    for line in file:
        scene_id = line.strip()

        if scene_id:
            val_scene_ids.add(scene_id)

# Prepare counters for valid and invalid bounding boxes.
statistics = {}

for split_name in ["Train", "Validation"]:
    for group_name in ["Included", "Excluded"]:
        statistics[(split_name, group_name)] = {
            "total_matching": 0,
            "valid_boxes": 0,
            "invalid_boxes": 0,
            "invalid_coordinates": 0,
            "category_counter": Counter()
        }

# Read all JSON annotation files.
for file_name in os.listdir(annotation_folder):

    if not file_name.endswith(".json"):
        continue

    # Get the scene ID from the filename.
    scene_id = os.path.splitext(file_name)[0]

    # Determine the official split for the current scene.
    if scene_id in train_scene_ids:
        split_name = "Train"

    elif scene_id in val_scene_ids:
        split_name = "Validation"

    else:
        continue

    # Load the annotation JSON.
    file_path = os.path.join(annotation_folder, file_name)

    with open(file_path, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)

    image_width = annotation_data.get("width", 0)
    image_height = annotation_data.get("height", 0)

    # Process each annotated object.
    for object_data in annotation_data.get("objects", []):

        category_name = object_data.get("label", "unknown")

        # Only analyze categories supported by the existing CNN models.
        if category_name not in cnn_class_names:
            continue

        properties = object_data.get("properties", {})

        if properties.get("included", True):
            group_name = "Included"
        else:
            group_name = "Excluded"

        current_stats = statistics[(split_name, group_name)]

        current_stats["total_matching"] += 1
        current_stats["category_counter"][category_name] += 1

        # Read the bounding box coordinates.
        bounding_box = object_data.get("bbox", {})

        try:
            xmin = float(bounding_box["xmin"])
            ymin = float(bounding_box["ymin"])
            xmax = float(bounding_box["xmax"])
            ymax = float(bounding_box["ymax"])

        except (KeyError, TypeError, ValueError):
            current_stats["invalid_boxes"] += 1
            continue

        # Check whether the box has positive width and height.
        if xmax <= xmin or ymax <= ymin:
            current_stats["invalid_boxes"] += 1
            continue

        # Check whether the box is inside the image boundaries.
        if (
            xmin < 0
            or ymin < 0
            or xmax > image_width
            or ymax > image_height
        ):
            current_stats["invalid_coordinates"] += 1
            continue

        # Count the box as valid if all checks pass.
        current_stats["valid_boxes"] += 1

# Print the final statistics.
for split_name in ["Train", "Validation"]:

    print()
    print("==========", split_name, "==========")

    for group_name in ["Included", "Excluded"]:

        current_stats = statistics[(split_name, group_name)]

        print()
        print("Group:", group_name)
        print("Matching objects:", current_stats["total_matching"])
        print("Valid bounding boxes:", current_stats["valid_boxes"])
        print("Invalid box shapes or missing coordinates:", current_stats["invalid_boxes"])
        print("Boxes outside image boundaries:", current_stats["invalid_coordinates"])

In [ ]:
# This code checks whether every official training and validation scene
# has a corresponding original image in the local image folders.
# It searches the existing image directories and records missing images.
# It only reads filenames and does not copy, delete, or modify any files.

import os

# Set the paths to the official MTSD split files.
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"
val_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\val.txt"

# List all folders that may contain the original MTSD images.
image_folders = [
    r".\data\raw_images\train_part_1\images",
    r".\data\raw_images\train_part_2\images",
    r".\data\raw_images\train_part_3\images",
    r".\data\raw_images\val\images"
]

# Define the image extensions that we want to check.
image_extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# Read the official training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = []

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            train_scene_ids.append(scene_id)

# Read the official validation scene IDs.
with open(val_split_path, "r", encoding="utf-8") as file:
    val_scene_ids = []

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            val_scene_ids.append(scene_id)

# Build a dictionary that records whether each scene image exists.
image_path_by_scene = {}

# Search each local image folder.
for image_folder in image_folders:

    print("Checking image folder:", image_folder)

    if not os.path.isdir(image_folder):
        print("Warning: This folder does not exist.")
        continue

    # Check each scene ID in the official dataset.
    for scene_id in train_scene_ids + val_scene_ids:

        # Skip scene IDs that have already been matched.
        if scene_id in image_path_by_scene:
            continue

        # Try each supported image extension.
        for image_extension in image_extensions:

            possible_image_path = os.path.join(
                image_folder,
                scene_id + image_extension
            )

            if os.path.isfile(possible_image_path):
                image_path_by_scene[scene_id] = possible_image_path
                break

# Find training images that could not be located.
missing_train_images = []

for scene_id in train_scene_ids:

    if scene_id not in image_path_by_scene:
        missing_train_images.append(scene_id)

# Find validation images that could not be located.
missing_val_images = []

for scene_id in val_scene_ids:

    if scene_id not in image_path_by_scene:
        missing_val_images.append(scene_id)

# Print the final results.
print()
print("========== Image Matching Results ==========")

print("Official training scenes:", len(train_scene_ids))
print("Training images found:", len(train_scene_ids) - len(missing_train_images))
print("Training images missing:", len(missing_train_images))

print()

print("Official validation scenes:", len(val_scene_ids))
print("Validation images found:", len(val_scene_ids) - len(missing_val_images))
print("Validation images missing:", len(missing_val_images))

print()
print("Total unique scene images found:", len(image_path_by_scene))

# Print a small sample of missing training images, if any.
if len(missing_train_images) > 0:

    print()
    print("First 20 missing training scene IDs:")

    for scene_id in missing_train_images[:20]:
        print(scene_id)

# Print a small sample of missing validation images, if any.
if len(missing_val_images) > 0:

    print()
    print("First 20 missing validation scene IDs:")

    for scene_id in missing_val_images[:20]:
        print(scene_id)

# Save the image matching results to a text file for later inspection.
report_path = r".\Detection_Inspection\image_matching_report.txt"

os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, "w", encoding="utf-8") as file:

    file.write("Official MTSD Image Matching Report\n")
    file.write("===================================\n\n")

    file.write("Training scenes: " + str(len(train_scene_ids)) + "\n")
    file.write("Training images found: " + str(len(train_scene_ids) - len(missing_train_images)) + "\n")
    file.write("Training images missing: " + str(len(missing_train_images)) + "\n\n")

    file.write("Validation scenes: " + str(len(val_scene_ids)) + "\n")
    file.write("Validation images found: " + str(len(val_scene_ids) - len(missing_val_images)) + "\n")
    file.write("Validation images missing: " + str(len(missing_val_images)) + "\n\n")

    file.write("Missing training scene IDs:\n")

    for scene_id in missing_train_images:
        file.write(scene_id + "\n")

    file.write("\nMissing validation scene IDs:\n")

    for scene_id in missing_val_images:
        file.write(scene_id + "\n")

print()
print("Matching report saved to:")
print(report_path)

In [ ]:
# This code creates an initial YOLO object detection dataset from the MTSD annotations.
# It uses the official training and validation splits and converts every accepted
# bounding box into one unified traffic_sign class.
# It skips ambiguous objects, objects outside the image frame, and invalid boxes.
# It does not delete or modify any original image, annotation, or CNN model file.
# Existing output folders are protected so that previous work is not overwritten.

import os
import json
import shutil
from PIL import Image

# Set the path to the original MTSD annotation folder.
annotation_folder = r".\data\raw\mtsd_v2_fully_annotated\annotations"

# Set the paths to the official training and validation split files.
train_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\train.txt"
val_split_path = r".\data\raw\mtsd_v2_fully_annotated\splits\val.txt"

# Set the local folders that contain the original MTSD images.
image_folders = [
    r".\data\raw_images\train_part_1\images",
    r".\data\raw_images\train_part_2\images",
    r".\data\raw_images\train_part_3\images",
    r".\data\raw_images\val\images"
]

# Set the output folder for the new YOLO dataset.
output_folder = r".\YOLO_Detection_Dataset"

# Stop if the output folder already exists.
# This prevents the code from overwriting a previous dataset.
if os.path.exists(output_folder):
    raise FileExistsError(
        "The output folder already exists. "
        "Please inspect it or choose a new output folder before running again:\n"
        + output_folder
    )

# Create the main folders for images and labels.
os.makedirs(os.path.join(output_folder, "images", "train"))
os.makedirs(os.path.join(output_folder, "images", "val"))
os.makedirs(os.path.join(output_folder, "labels", "train"))
os.makedirs(os.path.join(output_folder, "labels", "val"))

# Read the official training scene IDs.
with open(train_split_path, "r", encoding="utf-8") as file:
    train_scene_ids = []

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            train_scene_ids.append(scene_id)

# Read the official validation scene IDs.
with open(val_split_path, "r", encoding="utf-8") as file:
    val_scene_ids = []

    for line in file:
        scene_id = line.strip()

        if scene_id != "":
            val_scene_ids.append(scene_id)

# Define the image extensions supported by this script.
image_extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# Build a dictionary that maps each scene ID to its original image path.
image_path_by_scene = {}

for image_folder in image_folders:

    if not os.path.isdir(image_folder):
        continue

    for scene_id in train_scene_ids + val_scene_ids:

        if scene_id in image_path_by_scene:
            continue

        for image_extension in image_extensions:

            possible_path = os.path.join(
                image_folder,
                scene_id + image_extension
            )

            if os.path.isfile(possible_path):
                image_path_by_scene[scene_id] = possible_path
                break

# Create counters to record the dataset preparation results.
statistics = {
    "Train": {
        "images_processed": 0,
        "images_with_objects": 0,
        "images_without_objects": 0,
        "objects_written": 0,
        "ambiguous_skipped": 0,
        "out_of_frame_skipped": 0,
        "invalid_boxes_skipped": 0,
        "missing_images": 0,
        "unreadable_images": 0
    },
    "Validation": {
        "images_processed": 0,
        "images_with_objects": 0,
        "images_without_objects": 0,
        "objects_written": 0,
        "ambiguous_skipped": 0,
        "out_of_frame_skipped": 0,
        "invalid_boxes_skipped": 0,
        "missing_images": 0,
        "unreadable_images": 0
    }
}

# This helper function creates an image link or copies the image if necessary.
def add_image_to_dataset(source_path, destination_path):

    # Try to create a hard link first to reduce additional disk usage.
    try:
        os.link(source_path, destination_path)

    except OSError:
        # If a hard link cannot be created, copy the image instead.
        shutil.copy2(source_path, destination_path)

# Process the official training and validation splits separately.
splits_to_process = [
    ("Train", train_scene_ids, "train"),
    ("Validation", val_scene_ids, "val")
]

for split_name, scene_ids, output_split_name in splits_to_process:

    print()
    print("Starting dataset preparation for:", split_name)

    for image_number, scene_id in enumerate(scene_ids, start=1):

        # Find the original image.
        if scene_id not in image_path_by_scene:

            statistics[split_name]["missing_images"] += 1
            continue

        source_image_path = image_path_by_scene[scene_id]

        # Find the matching annotation JSON file.
        annotation_path = os.path.join(
            annotation_folder,
            scene_id + ".json"
        )

        if not os.path.isfile(annotation_path):
            print("Warning: Annotation file not found:", scene_id)
            continue

        # Load the annotation information.
        with open(annotation_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        # Read the original image dimensions from the JSON file.
        json_width = annotation_data.get("width", 0)
        json_height = annotation_data.get("height", 0)

        # Open the image to confirm that it can be read.
        try:
            with Image.open(source_image_path) as image:
                actual_width, actual_height = image.size

        except Exception:
            statistics[split_name]["unreadable_images"] += 1
            continue

        # Use actual image dimensions for safe coordinate conversion.
        image_width = actual_width
        image_height = actual_height

        # Print a warning if the JSON dimensions disagree with the image.
        if json_width != image_width or json_height != image_height:
            print(
                "Warning: Image size differs from annotation size:",
                scene_id,
                "JSON:",
                json_width,
                json_height,
                "Image:",
                image_width,
                image_height
            )

        # Store the YOLO labels for the current image.
        yolo_label_lines = []

        # Process every annotated object in the image.
        for object_data in annotation_data.get("objects", []):

            properties = object_data.get("properties", {})

            # Skip objects whose annotations are explicitly ambiguous.
            if properties.get("ambiguous", False):
                statistics[split_name]["ambiguous_skipped"] += 1
                continue

            # Skip objects that are explicitly marked as out of frame.
            if properties.get("out-of-frame", False):
                statistics[split_name]["out_of_frame_skipped"] += 1
                continue

            # Read the bounding box coordinates.
            bounding_box = object_data.get("bbox", {})

            try:
                xmin = float(bounding_box["xmin"])
                ymin = float(bounding_box["ymin"])
                xmax = float(bounding_box["xmax"])
                ymax = float(bounding_box["ymax"])

            except (KeyError, TypeError, ValueError):
                statistics[split_name]["invalid_boxes_skipped"] += 1
                continue

            # Check whether the bounding box has positive dimensions.
            if xmax <= xmin or ymax <= ymin:
                statistics[split_name]["invalid_boxes_skipped"] += 1
                continue

            # Skip boxes that extend beyond the actual image boundaries.
            if (
                xmin < 0
                or ymin < 0
                or xmax > image_width
                or ymax > image_height
            ):
                statistics[split_name]["invalid_boxes_skipped"] += 1
                continue

            # Convert the bounding box into YOLO normalized coordinates.
            box_center_x = ((xmin + xmax) / 2.0) / image_width
            box_center_y = ((ymin + ymax) / 2.0) / image_height

            box_width = (xmax - xmin) / image_width
            box_height = (ymax - ymin) / image_height

            # YOLO requires the class ID followed by normalized box coordinates.
            # All accepted objects use class 0, meaning traffic_sign.
            label_line = (
                "0 "
                + str(box_center_x)
                + " "
                + str(box_center_y)
                + " "
                + str(box_width)
                + " "
                + str(box_height)
            )

            yolo_label_lines.append(label_line)

        # Set the output paths for the image and its label file.
        output_image_folder = os.path.join(
            output_folder,
            "images",
            output_split_name
        )

        output_label_folder = os.path.join(
            output_folder,
            "labels",
            output_split_name
        )

        # Preserve the original image extension.
        image_extension = os.path.splitext(source_image_path)[1]

        destination_image_path = os.path.join(
            output_image_folder,
            scene_id + image_extension
        )

        destination_label_path = os.path.join(
            output_label_folder,
            scene_id + ".txt"
        )

        # Add the original image to the new dataset.
        add_image_to_dataset(
            source_image_path,
            destination_image_path
        )

        # Write the YOLO label file.
        # An empty file is created when no object passes the filtering rules.
        with open(destination_label_path, "w", encoding="utf-8") as file:
            for label_line in yolo_label_lines:
                file.write(label_line + "\n")

        # Update the statistics.
        statistics[split_name]["images_processed"] += 1
        statistics[split_name]["objects_written"] += len(yolo_label_lines)

        if len(yolo_label_lines) > 0:
            statistics[split_name]["images_with_objects"] += 1
        else:
            statistics[split_name]["images_without_objects"] += 1

        # Print progress after every 1000 images.
        if image_number % 1000 == 0:
            print(
                split_name,
                "processed:",
                image_number,
                "/",
                len(scene_ids)
            )

# Create the YOLO dataset configuration file.
dataset_yaml_path = os.path.join(output_folder, "dataset.yaml")

yaml_content = (
    "path: " + output_folder.replace("\\", "/") + "\n"
    + "train: images/train\n"
    + "val: images/val\n"
    + "\n"
    + "names:\n"
    + "  0: traffic_sign\n"
)

with open(dataset_yaml_path, "w", encoding="utf-8") as file:
    file.write(yaml_content)

# Print the final dataset preparation report.
print()
print("========== YOLO Dataset Preparation Summary ==========")

for split_name in ["Train", "Validation"]:

    print()
    print("Split:", split_name)

    for statistic_name, statistic_value in statistics[split_name].items():
        print(statistic_name + ":", statistic_value)

print()
print("YOLO dataset folder:")
print(output_folder)

print()
print("YOLO configuration file:")
print(dataset_yaml_path)

print()
print("Dataset preparation completed.")

In [ ]:

# This code checks whether the YOLO bounding boxes are correctly placed on the original images.

import os
import random
import cv2
import matplotlib.pyplot as plt

# Step 1: Define the YOLO dataset folder.
dataset_folder = r".\YOLO_Detection_Dataset"

# Step 2: Define the training image folder and label folder.
train_image_folder = os.path.join(dataset_folder, "images", "train")
train_label_folder = os.path.join(dataset_folder, "labels", "train")

# Step 3: Get all image file names from the training folder.
all_image_names = []

for image_name in os.listdir(train_image_folder):
    if image_name.lower().endswith((".jpg", ".jpeg", ".png")):
        all_image_names.append(image_name)

# Step 4: Select five random images for visual inspection.
number_of_images_to_show = 5

if len(all_image_names) < number_of_images_to_show:
    number_of_images_to_show = len(all_image_names)

selected_image_names = random.sample(
    all_image_names,
    number_of_images_to_show
)

# Step 5: Create a figure for displaying the selected images.
plt.figure(figsize=(18, 12))

# Step 6: Read each image and draw the corresponding bounding boxes.
for image_index in range(len(selected_image_names)):

    image_name = selected_image_names[image_index]

    image_path = os.path.join(
        train_image_folder,
        image_name
    )

    label_name = os.path.splitext(image_name)[0] + ".txt"

    label_path = os.path.join(
        train_label_folder,
        label_name
    )

    # Read the original image using OpenCV.
    image = cv2.imread(image_path)

    if image is None:
        print("Warning: Could not read image:", image_path)
        continue

    # Convert the image from BGR format to RGB format.
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Get the actual image height and width.
    image_height, image_width = image.shape[:2]

    number_of_boxes = 0

    # Step 7: Read the YOLO label file if it exists.
    if os.path.exists(label_path):

        with open(label_path, "r", encoding="utf-8") as label_file:

            label_lines = label_file.readlines()

        # Step 8: Convert normalized YOLO coordinates into pixel coordinates.
        for label_line in label_lines:

            label_line = label_line.strip()

            if label_line == "":
                continue

            label_values = label_line.split()

            if len(label_values) != 5:
                print("Warning: Incorrect label format:", label_path)
                continue

            class_id = int(label_values[0])

            x_center_normalized = float(label_values[1])
            y_center_normalized = float(label_values[2])
            box_width_normalized = float(label_values[3])
            box_height_normalized = float(label_values[4])

            x_center = x_center_normalized * image_width
            y_center = y_center_normalized * image_height

            box_width = box_width_normalized * image_width
            box_height = box_height_normalized * image_height

            x_min = int(x_center - box_width / 2)
            y_min = int(y_center - box_height / 2)

            x_max = int(x_center + box_width / 2)
            y_max = int(y_center + box_height / 2)

            # Step 9: Draw the bounding box on the image.
            cv2.rectangle(
                image,
                (x_min, y_min),
                (x_max, y_max),
                (255, 0, 0),
                2
            )

            # Add a label above the bounding box.
            cv2.putText(
                image,
                "traffic_sign",
                (x_min, max(y_min - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 0, 0),
                1
            )

            number_of_boxes = number_of_boxes + 1

    # Step 10: Display the image and its bounding boxes.
    plt.subplot(2, 3, image_index + 1)

    plt.imshow(image)

    plt.title(
        "Image: " + image_name[:18]
        + "\nBounding boxes: " + str(number_of_boxes)
    )

    plt.axis("off")

# Step 11: Display all selected images.
plt.tight_layout()
plt.show()


In [ ]:

# This code installs the Ultralytics package for training the YOLO object detection model.

import sys
import subprocess

# Step 1: Get the Python executable used by the current notebook.
python_executable = sys.executable

print("Python environment:")
print(python_executable)

# Step 2: Install the Ultralytics package into this Python environment.
subprocess.check_call([
    python_executable,
    "-m",
    "pip",
    "install",
    "ultralytics"
])

# Step 3: Print a message after the installation finishes.
print("Ultralytics installation completed.")


In [ ]:

# This code checks the YOLO installation, PyTorch version, CUDA availability, and GPU information before training.

import sys
import torch
import ultralytics

# Step 1: Display the Python environment.
print("Python executable:")
print(sys.executable)

# Step 2: Display the installed Ultralytics version.
print("\nUltralytics version:")
print(ultralytics.__version__)

# Step 3: Display the installed PyTorch version.
print("\nPyTorch version:")
print(torch.__version__)

# Step 4: Check whether CUDA is available.
print("\nCUDA available:")
print(torch.cuda.is_available())

# Step 5: Display the CUDA version used by PyTorch.
print("\nPyTorch CUDA version:")
print(torch.version.cuda)

# Step 6: Display the GPU information if CUDA is available.
if torch.cuda.is_available():

    print("\nGPU name:")
    print(torch.cuda.get_device_name(0))

    print("\nGPU memory:")
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(round(gpu_memory_gb, 2), "GB")

    print("\nGPU is ready for training.")

else:

    print("\nCUDA is not available.")
    print("The model would need to use the CPU until the GPU environment is fixed.")


In [ ]:

# This code trains a YOLO11n object detection model to detect traffic signs in full road images.

import os
import sys
import torch
from ultralytics import YOLO

# Step 1: Define the main project folder.
project_folder = r"."

# Step 2: Define the YOLO dataset configuration file.
dataset_yaml_path = os.path.join(
    project_folder,
    "YOLO_Detection_Dataset",
    "dataset.yaml"
)

# Step 3: Define the folder where the training results will be saved.
results_folder = os.path.join(
    project_folder,
    "YOLO_Detection_Results"
)

# Step 4: Check whether the dataset configuration file exists.
if not os.path.exists(dataset_yaml_path):
    raise FileNotFoundError(
        "The dataset.yaml file was not found: "
        + dataset_yaml_path
    )

# Step 5: Display the current training environment.
print("Python executable:")
print(sys.executable)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:")
    print(torch.cuda.get_device_name(0))

# Step 6: Display the dataset configuration path.
print("\nDataset configuration:")
print(dataset_yaml_path)

# Step 7: Display the training results folder.
print("\nTraining results folder:")
print(results_folder)

# Step 8: Load the lightweight YOLO11n pretrained model.
# The pretrained weights help the model learn faster than training from scratch.
model = YOLO("yolo11n.pt")

# Step 9: Start training the traffic sign detector.
training_results = model.train(
    data=dataset_yaml_path,
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=4,
    patience=10,
    project=results_folder,
    name="yolo11n_traffic_sign",
    exist_ok=False,
    pretrained=True,
    amp=True,
    plots=True,
    save=True,
    verbose=True
)

# Step 10: Print a message after training finishes.
print("\nYOLO training process finished.")

# Step 11: Display the location of the saved training results.
print("Training results are saved in:")
print(
    os.path.join(
        results_folder,
        "yolo11n_traffic_sign"
    )
)


In [ ]:

# This code checks the environment, verifies the YOLO dataset, loads the pretrained YOLO11n model, and trains a traffic sign detector.
# The data loader uses workers=0 to avoid the previous Windows multiprocessing error.

# ============================================================
# Step 1: Import the required Python libraries.
# ============================================================

import os
import sys
import torch
import ultralytics

from ultralytics import YOLO


# ============================================================
# Step 2: Display the Python and deep learning environment.
# ============================================================

print("=" * 60)
print("STEP 1: CHECKING THE PYTHON ENVIRONMENT")
print("=" * 60)

print("Python executable:")
print(sys.executable)

print("\nUltralytics version:")
print(ultralytics.__version__)

print("\nPyTorch version:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():

    print("\nGPU name:")
    print(torch.cuda.get_device_name(0))

    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print("\nGPU memory:")
    print(round(gpu_memory_gb, 2), "GB")

else:

    raise RuntimeError(
        "CUDA is not available. Please check the GPU environment before training."
    )


# ============================================================
# Step 3: Define the project and dataset paths.
# ============================================================

print("\n" + "=" * 60)
print("STEP 2: CHECKING THE YOLO DATASET")
print("=" * 60)

project_folder = r"."

dataset_folder = os.path.join(
    project_folder,
    "YOLO_Detection_Dataset"
)

dataset_yaml_path = os.path.join(
    dataset_folder,
    "dataset.yaml"
)

train_image_folder = os.path.join(
    dataset_folder,
    "images",
    "train"
)

train_label_folder = os.path.join(
    dataset_folder,
    "labels",
    "train"
)

val_image_folder = os.path.join(
    dataset_folder,
    "images",
    "val"
)

val_label_folder = os.path.join(
    dataset_folder,
    "labels",
    "val"
)

print("Dataset folder:")
print(dataset_folder)

print("\nDataset configuration file:")
print(dataset_yaml_path)


# ============================================================
# Step 4: Verify that all required dataset folders exist.
# ============================================================

required_paths = [
    dataset_yaml_path,
    train_image_folder,
    train_label_folder,
    val_image_folder,
    val_label_folder
]

for current_path in required_paths:

    if not os.path.exists(current_path):

        raise FileNotFoundError(
            "Required dataset path was not found: "
            + current_path
        )

    else:

        print("Path exists:", current_path)


# ============================================================
# Step 5: Check the YOLO dataset configuration.
# ============================================================

print("\n" + "=" * 60)
print("STEP 3: READING THE DATASET CONFIGURATION")
print("=" * 60)

with open(
    dataset_yaml_path,
    "r",
    encoding="utf-8"
) as dataset_file:

    dataset_configuration_text = dataset_file.read()

print(dataset_configuration_text)


# ============================================================
# Step 6: Count the training and validation images.
# ============================================================

print("\n" + "=" * 60)
print("STEP 4: COUNTING THE DATASET FILES")
print("=" * 60)

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

train_image_names = []

for image_name in os.listdir(train_image_folder):

    if image_name.lower().endswith(image_extensions):

        train_image_names.append(image_name)


val_image_names = []

for image_name in os.listdir(val_image_folder):

    if image_name.lower().endswith(image_extensions):

        val_image_names.append(image_name)


print("Number of training images:")
print(len(train_image_names))

print("\nNumber of validation images:")
print(len(val_image_names))

if len(train_image_names) == 0:

    raise RuntimeError(
        "No training images were found."
    )

if len(val_image_names) == 0:

    raise RuntimeError(
        "No validation images were found."
    )


# ============================================================
# Step 7: Display the training configuration.
# ============================================================

print("\n" + "=" * 60)
print("STEP 5: PREPARING THE TRAINING SETTINGS")
print("=" * 60)

model_name = "yolo11n.pt"

number_of_epochs = 50

image_size = 640

batch_size = 8

number_of_workers = 0

early_stopping_patience = 10

training_device = 0

results_folder = os.path.join(
    project_folder,
    "YOLO_Detection_Results"
)

experiment_name = "yolo11n_traffic_sign_retry"

experiment_folder = os.path.join(
    results_folder,
    experiment_name
)

print("Model:", model_name)

print("Epochs:", number_of_epochs)

print("Image size:", image_size)

print("Batch size:", batch_size)

print("Data loader workers:", number_of_workers)

print("Early stopping patience:", early_stopping_patience)

print("Training device:", training_device)

print("Experiment name:", experiment_name)

print("Experiment folder:", experiment_folder)


# ============================================================
# Step 8: Prevent accidental overwriting of an existing run.
# ============================================================

if os.path.exists(experiment_folder):

    raise FileExistsError(
        "This experiment folder already exists:\n"
        + experiment_folder
        + "\n\nPlease change experiment_name to a new name before training."
    )


# ============================================================
# Step 9: Load the pretrained YOLO11n model.
# ============================================================

print("\n" + "=" * 60)
print("STEP 6: LOADING THE PRETRAINED YOLO MODEL")
print("=" * 60)

model = YOLO(model_name)

print("YOLO model loaded successfully.")


# ============================================================
# Step 10: Start training the traffic sign detector.
# ============================================================

print("\n" + "=" * 60)
print("STEP 7: STARTING YOLO TRAINING")
print("=" * 60)

training_results = model.train(

    # Dataset configuration.
    data=dataset_yaml_path,

    # Number of training epochs.
    epochs=number_of_epochs,

    # Input image size.
    imgsz=image_size,

    # Number of images processed in one batch.
    batch=batch_size,

    # Use the first CUDA GPU.
    device=training_device,

    # Disable multiprocessing in the data loader.
    workers=number_of_workers,

    # Stop early if validation performance does not improve.
    patience=early_stopping_patience,

    # Save results in a separate project folder.
    project=results_folder,

    # Use a new experiment name.
    name=experiment_name,

    # Do not overwrite an existing experiment.
    exist_ok=False,

    # Use pretrained model weights.
    pretrained=True,

    # Enable automatic mixed precision.
    amp=True,

    # Generate training plots.
    plots=True,

    # Save model checkpoints.
    save=True,

    # Display training information.
    verbose=True
)


# ============================================================
# Step 11: Display the final training output paths.
# ============================================================

print("\n" + "=" * 60)
print("STEP 8: TRAINING FINISHED")
print("=" * 60)

best_model_path = os.path.join(
    experiment_folder,
    "weights",
    "best.pt"
)

last_model_path = os.path.join(
    experiment_folder,
    "weights",
    "last.pt"
)

print("Best model path:")
print(best_model_path)

print("\nLast model path:")
print(last_model_path)

print("\nBest model exists:")
print(os.path.exists(best_model_path))

print("\nLast model exists:")
print(os.path.exists(last_model_path))

print("\nYOLO training script completed.")


In [ ]:

# This code checks whether YOLO training checkpoints have been saved.
# It does not stop or modify the current training process.

import os

# Step 1: Define the current YOLO experiment folder.
experiment_folder = (
    r"."
    r"\YOLO_Detection_Results"
    r"\yolo11n_traffic_sign_retry"
)

# Step 2: Define the folder containing the saved model weights.
weights_folder = os.path.join(
    experiment_folder,
    "weights"
)

# Step 3: Check whether the weights folder exists.
print("Weights folder exists:")
print(os.path.exists(weights_folder))

# Step 4: Check the last checkpoint.
last_model_path = os.path.join(
    weights_folder,
    "last.pt"
)

print("\nLast checkpoint exists:")
print(os.path.exists(last_model_path))

# Step 5: Check the best checkpoint.
best_model_path = os.path.join(
    weights_folder,
    "best.pt"
)

print("\nBest checkpoint exists:")
print(os.path.exists(best_model_path))

# Step 6: Display the checkpoint file sizes if they exist.
if os.path.exists(last_model_path):

    last_size_mb = os.path.getsize(last_model_path) / (1024 ** 2)

    print("\nLast checkpoint size:")
    print(round(last_size_mb, 2), "MB")

if os.path.exists(best_model_path):

    best_size_mb = os.path.getsize(best_model_path) / (1024 ** 2)

    print("\nBest checkpoint size:")
    print(round(best_size_mb, 2), "MB")


In [ ]:
# This code resumes the previous YOLO training session from the last saved checkpoint.
# It should be run after the original training cell has been stopped and the checkpoint has been checked.

from ultralytics import YOLO

# Load the most recently saved training checkpoint.
last_checkpoint_path = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry\weights\last.pt"

model = YOLO(last_checkpoint_path)

# Resume the previous training session, including its saved training state when available.
results = model.train(resume=True)

In [ ]:
# This code checks the saved YOLO checkpoint before resuming the interrupted training.
# It verifies that the last checkpoint exists and then loads the saved model.

import os
from ultralytics import YOLO

# Set the full path of the last checkpoint saved during the previous training session.
last_checkpoint_path = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry\weights\last.pt"

# Check whether the checkpoint file exists before trying to load it.
if os.path.exists(last_checkpoint_path):

    # Print a message to confirm that the checkpoint file was found.
    print("The last checkpoint file was found.")

    # Print the file size so that we can check whether the file is present and non-empty.
    checkpoint_size_mb = os.path.getsize(last_checkpoint_path) / (1024 * 1024)
    print("Checkpoint file size:", round(checkpoint_size_mb, 2), "MB")

    # Load the saved YOLO model from the checkpoint file.
    model = YOLO(last_checkpoint_path)

    # Print a message confirming that the checkpoint was loaded.
    print("The YOLO checkpoint was loaded successfully.")

else:

    # Print a warning if the checkpoint file cannot be found.
    print("The checkpoint file was not found. Please check the file path.")

In [ ]:
# This code resumes the previous YOLO11n traffic sign detection training session.
# It uses the last saved checkpoint instead of starting a completely new training session.

from ultralytics import YOLO

# Set the path to the last checkpoint saved during the previous training session.
last_checkpoint_path = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry\weights\last.pt"

# Load the YOLO model and its saved checkpoint.
model = YOLO(last_checkpoint_path)

# Resume the previous training session using the saved optimizer and training information.
# Ultralytics will try to continue from the saved epoch and use the previous training configuration.
results = model.train(resume=True)

In [ ]:
# This code checks whether the Python environment and NVIDIA GPU are working after the kernel restarted.
# It does not start or resume any model training.

import os
import torch
from ultralytics import YOLO

# Print a message to confirm that the notebook kernel is running again.
print("The Jupyter kernel is running.")

# Check whether PyTorch can detect CUDA.
print("CUDA available:", torch.cuda.is_available())

# If CUDA is available, print the GPU name and current memory information.
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    # Read the amount of GPU memory currently allocated to PyTorch.
    allocated_memory_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)

    # Read the amount of GPU memory currently reserved by PyTorch.
    reserved_memory_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

    print("Allocated GPU memory:", round(allocated_memory_gb, 2), "GB")
    print("Reserved GPU memory:", round(reserved_memory_gb, 2), "GB")

# Check whether the saved checkpoint still exists.
last_checkpoint_path = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry\weights\last.pt"

print("Last checkpoint exists:", os.path.exists(last_checkpoint_path))

# Print the saved checkpoint size if the file exists.
if os.path.exists(last_checkpoint_path):
    checkpoint_size_mb = os.path.getsize(last_checkpoint_path) / (1024 * 1024)
    print("Last checkpoint size:", round(checkpoint_size_mb, 2), "MB")

In [ ]:
# This code checks the saved YOLO training files and training history.
# It helps us determine whether any training epochs were completed before the interruption.

import os
import pandas as pd

# Set the folder where the previous YOLO training results were saved.
training_folder = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry"

# Set the folder where the saved model checkpoints are stored.
weights_folder = os.path.join(training_folder, "weights")

# Print a heading so that the output is easy to understand.
print("Checking the previous YOLO training folder...")
print("Training folder exists:", os.path.exists(training_folder))
print("Weights folder exists:", os.path.exists(weights_folder))

# List the files in the weights folder and show their sizes and modification times.
if os.path.exists(weights_folder):
    print("\nSaved checkpoint files:")

    for file_name in os.listdir(weights_folder):
        file_path = os.path.join(weights_folder, file_name)

        if os.path.isfile(file_path):
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            file_modified_time = os.path.getmtime(file_path)

            print(
                file_name,
                "| Size:", round(file_size_mb, 2), "MB",
                "| Modified timestamp:", file_modified_time
            )

# Check whether the training history CSV file exists.
results_csv_path = os.path.join(training_folder, "results.csv")

if os.path.exists(results_csv_path):
    print("\nTraining history file found.")

    # Read the saved training history without loading the model.
    training_history = pd.read_csv(results_csv_path)

    # Display the number of recorded rows and the latest recorded training results.
    print("Number of recorded rows:", len(training_history))
    print("\nLatest training history:")
    print(training_history.tail(3).to_string(index=False))

else:
    print("\nNo results.csv file was found yet.")
    print("The training may have stopped before the first epoch was recorded.")

In [ ]:
# Step 1: Check the dataset folders and locate the validation images.

import os

# Set the main dataset folder.
dataset_folder = r".\YOLO_Detection_Dataset"

# Print the dataset folder path.
print("Checking the dataset folder...")
print("Dataset folder:", dataset_folder)

# Check whether the dataset folder exists.
if os.path.exists(dataset_folder):
    print("Dataset folder exists: Yes")

    # Display the folders and files inside the dataset folder.
    print("\nFiles and folders inside the dataset folder:")

    for item in os.listdir(dataset_folder):
        print(item)

else:
    print("Dataset folder exists: No")
    print("Please check the dataset path.")

In [ ]:
# Step 2: Check the image and label folders to locate the validation dataset.

import os

# Set the paths of the image and label folders.
dataset_folder = r".\YOLO_Detection_Dataset"

images_folder = os.path.join(dataset_folder, "images")
labels_folder = os.path.join(dataset_folder, "labels")

# Check the image folder.
print("Checking the images folder...")
print("Images folder:", images_folder)

if os.path.exists(images_folder):

    # Display the subfolders inside the images folder.
    for item in os.listdir(images_folder):

        item_path = os.path.join(images_folder, item)

        if os.path.isdir(item_path):

            # Count the files inside this subfolder.
            number_of_files = len(os.listdir(item_path))

            print("Image subfolder:", item)
            print("Number of files:", number_of_files)

        else:
            print("File inside images folder:", item)

else:
    print("Images folder was not found.")

# Check the label folder.
print("\nChecking the labels folder...")
print("Labels folder:", labels_folder)

if os.path.exists(labels_folder):

    # Display the subfolders inside the labels folder.
    for item in os.listdir(labels_folder):

        item_path = os.path.join(labels_folder, item)

        if os.path.isdir(item_path):

            # Count the files inside this subfolder.
            number_of_files = len(os.listdir(item_path))

            print("Label subfolder:", item)
            print("Number of files:", number_of_files)

        else:
            print("File inside labels folder:", item)

else:
    print("Labels folder was not found.")

In [ ]:
# Step 3: Load the best saved YOLO model for evaluation.

import os
import torch
from ultralytics import YOLO

# Set the folder where the YOLO training results are saved.
training_folder = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry"

# Set the path of the best model checkpoint.
best_model_path = os.path.join(training_folder, "weights", "best.pt")

# Check whether the best model file exists.
print("Checking the best model file...")
print("Model path:", best_model_path)
print("Model file exists:", os.path.exists(best_model_path))

# Load the trained YOLO model.
if os.path.exists(best_model_path):

    # Load the saved model weights.
    best_model = YOLO(best_model_path)

    print("\nModel loaded successfully.")

    # Display the model task and model information.
    print("Model task:", best_model.task)
    print("Number of classes:", len(best_model.names))
    print("Class names:", best_model.names)

    # Display whether CUDA is available for prediction.
    print("CUDA available:", torch.cuda.is_available())

else:
    print("The best model file was not found.")
    print("Please check the model path.")

In [ ]:
# Step 4: Run predictions on 30 randomly selected validation images.

import os
import random
from ultralytics import YOLO

# Set the validation image folder.
validation_images_folder = r".\YOLO_Detection_Dataset\images\val"

# Set the folder where prediction images will be saved.
prediction_output_folder = r".\YOLO_Detection_Results\validation_predictions_30"

# Find all image files in the validation folder.
validation_image_files = []

for filename in os.listdir(validation_images_folder):

    # Check whether the file is an image.
    if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):

        # Add the complete image path to the list.
        full_image_path = os.path.join(validation_images_folder, filename)
        validation_image_files.append(full_image_path)

print("Total validation images found:", len(validation_image_files))

# Set a random seed so that the selection can be reproduced.
random.seed(42)

# Select up to 30 validation images.
number_of_images_to_test = min(30, len(validation_image_files))
selected_image_files = random.sample(validation_image_files, number_of_images_to_test)

print("Number of images selected:", len(selected_image_files))

# Run prediction using the already loaded best model.
prediction_results = best_model.predict(
    source=selected_image_files,
    imgsz=640,
    conf=0.25,
    save=True,
    project=r".\YOLO_Detection_Results",
    name="validation_predictions_30",
    exist_ok=True,
    verbose=True
)

# Display the folder containing the prediction images.
print("\nPrediction finished.")
print("Prediction images saved to:", prediction_output_folder)

In [ ]:
# This code evaluates the trained YOLO model on the complete validation dataset.
# It calculates the main object detection metrics without training the model again.

import os
from ultralytics import YOLO

# Step 1: Set the path of the trained model.
training_folder = r".\YOLO_Detection_Results\yolo11n_traffic_sign_retry"
best_model_path = os.path.join(training_folder, "weights", "best.pt")

# Step 2: Set the path of the dataset configuration file.
dataset_yaml_path = r".\YOLO_Detection_Dataset\dataset.yaml"

# Step 3: Load the best model that we trained before.
best_model = YOLO(best_model_path)

# Step 4: Evaluate the model on the validation dataset.
# The confidence threshold is kept at the default evaluation setting.
validation_metrics = best_model.val(
    data=dataset_yaml_path,
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    plots=True,
    project=r".\YOLO_Detection_Results",
    name="validation_full_evaluation",
    exist_ok=True
)

# Step 5: Print the main evaluation results.
print("Validation evaluation is finished.")

print("Precision:", validation_metrics.box.mp)
print("Recall:", validation_metrics.box.mr)
print("mAP at IoU 0.50:", validation_metrics.box.map50)
print("mAP at IoU 0.50 to 0.95:", validation_metrics.box.map)

In [ ]:
# This code searches the project folder for previously trained CNN model files.
# It also looks for class mapping files that tell us what each output class means.
# This code only searches for files and does not change or train any model.

import os

# Step 1: Set the main folder of our traffic sign project.
project_folder = r"."

# Step 2: Create empty lists to store the model files and class mapping files.
possible_model_files = []
possible_class_mapping_files = []

# Step 3: Define the file extensions that trained models may use.
model_file_extensions = (
    ".pt",
    ".pth",
    ".h5",
    ".keras",
    ".ckpt",
    ".pkl"
)

# Step 4: Walk through the project folder and its subfolders.
for current_folder, subfolders, filenames in os.walk(project_folder):

    # Skip folders that contain temporary files or prediction results.
    subfolders[:] = [
        folder_name
        for folder_name in subfolders
        if folder_name not in [
            "__pycache__",
            ".ipynb_checkpoints"
        ]
    ]

    # Step 5: Check every file in the current folder.
    for filename in filenames:

        # Create the complete path of the current file.
        full_file_path = os.path.join(current_folder, filename)

        # Check whether the file may contain trained model weights.
        if filename.lower().endswith(model_file_extensions):
            possible_model_files.append(full_file_path)

        # Check whether the file may contain class names or class indexes.
        if (
            "class" in filename.lower()
            or "label" in filename.lower()
            or "mapping" in filename.lower()
        ) and filename.lower().endswith(".json"):
            possible_class_mapping_files.append(full_file_path)

# Step 6: Print all model files that were found.
print("========== POSSIBLE TRAINED MODEL FILES ==========")

if len(possible_model_files) > 0:
    for model_file in possible_model_files:
        print(model_file)
else:
    print("No model files were found in this project folder.")

# Step 7: Print all possible class mapping files.
print()
print("========== POSSIBLE CLASS MAPPING FILES ==========")

if len(possible_class_mapping_files) > 0:
    for mapping_file in possible_class_mapping_files:
        print(mapping_file)
else:
    print("No class mapping JSON files were found.")

In [ ]:
# This code checks the saved CNN checkpoint and the class mapping files.
# It also searches for Python scripts and notebooks that may contain the original CNN architecture.
# This code only reads existing files and does not modify or train any model.

import os
import json
import torch

# Step 1: Set the paths of the CNN model and class mapping files.
project_folder = r"."

cnn_model_path = os.path.join(
    project_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_best.pth"
)

class_mapping_path = os.path.join(
    project_folder,
    "train",
    "final_class_to_index.json"
)

# Step 2: Check whether the required files exist.
print("========== FILE CHECK ==========")
print("CNN model exists:", os.path.isfile(cnn_model_path))
print("Class mapping exists:", os.path.isfile(class_mapping_path))

# Step 3: Load the CNN checkpoint onto the CPU for inspection.
# The checkpoint is assumed to be a file created by our own training process.
if os.path.isfile(cnn_model_path):
    cnn_checkpoint = torch.load(
        cnn_model_path,
        map_location="cpu",
        weights_only=False
    )

    print("\n========== CNN CHECKPOINT INFORMATION ==========")
    print("Checkpoint type:", type(cnn_checkpoint))

    # Check whether the checkpoint is a dictionary containing model information.
    if isinstance(cnn_checkpoint, dict):
        print("Checkpoint keys:", list(cnn_checkpoint.keys()))

        # Print a few values that may describe the training setup.
        for key in cnn_checkpoint.keys():
            if key not in ["state_dict", "model_state_dict", "model"]:
                value = cnn_checkpoint[key]

                if isinstance(value, (str, int, float, bool, list, tuple)):
                    print(key, ":", str(value)[:300])

        # Try to find the saved model weights.
        possible_weight_keys = [
            "model_state_dict",
            "state_dict",
            "model"
        ]

        cnn_state_dict = None

        for key in possible_weight_keys:
            if key in cnn_checkpoint and isinstance(cnn_checkpoint[key], dict):
                cnn_state_dict = cnn_checkpoint[key]
                print("\nModel weights found under key:", key)
                break

        # Some checkpoints directly contain the model state dictionary.
        if cnn_state_dict is None:
            if all(torch.is_tensor(value) for value in cnn_checkpoint.values()):
                cnn_state_dict = cnn_checkpoint
                print("\nThe checkpoint appears to directly contain model weights.")

        # Print the names and shapes of some saved model parameters.
        if cnn_state_dict is not None:
            print("\n========== MODEL WEIGHT SHAPES ==========")

            for index, (parameter_name, parameter_tensor) in enumerate(cnn_state_dict.items()):
                print(parameter_name, ":", tuple(parameter_tensor.shape))

                if index >= 14:
                    print("Only the first 15 parameter entries are shown.")
                    break
        else:
            print("The model weights were not found in the expected dictionary format.")

    else:
        print("The checkpoint is not a dictionary. We need to inspect its format.")

# Step 4: Read the class mapping JSON file.
print("\n========== CLASS MAPPING INFORMATION ==========")

if os.path.isfile(class_mapping_path):
    with open(class_mapping_path, "r", encoding="utf-8") as mapping_file:
        class_mapping = json.load(mapping_file)

    print("Mapping type:", type(class_mapping))

    if isinstance(class_mapping, dict):
        print("Number of mapping entries:", len(class_mapping))
        print("First 15 mapping entries:")

        for index, (class_name, class_index) in enumerate(class_mapping.items()):
            print(class_name, ":", class_index)

            if index >= 14:
                break
    else:
        print("Mapping preview:", str(class_mapping)[:1000])

# Step 5: Search for Python files and Jupyter notebooks in the project.
print("\n========== POSSIBLE CNN TRAINING CODE ==========")

possible_code_files = []

for current_folder, subfolders, filenames in os.walk(project_folder):

    # Skip folders that usually contain temporary or generated files.
    subfolders[:] = [
        folder_name
        for folder_name in subfolders
        if folder_name not in [
            "__pycache__",
            ".ipynb_checkpoints"
        ]
    ]

    for filename in filenames:
        if filename.lower().endswith((".py", ".ipynb")):
            full_file_path = os.path.join(current_folder, filename)
            possible_code_files.append(full_file_path)

for code_file in possible_code_files:
    print(code_file)

print("\nInspection finished. No training was started.")

In [ ]:
# This code reconstructs the CNN architecture based on the saved parameter shapes.
# It then loads the previously trained weights and checks whether the model is compatible.
# This code does not train the model or update any parameters.

import os
import torch
import torch.nn as nn

# Step 1: Set the path of the CNN model that we want to use.
project_folder = r"."

cnn_model_path = os.path.join(
    project_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_best.pth"
)

# Step 2: Select the device that will be used to run the CNN.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Selected device:", device)

# Step 3: Define the CNN architecture using the parameter shapes we found.
# The three convolutional layers have 32, 64, and 128 output channels.
# Three pooling operations would reduce a 64 by 64 image to an 8 by 8 feature map.

class TrafficSignCNN(nn.Module):

    def __init__(self, number_of_classes=326):
        super(TrafficSignCNN, self).__init__()

        # Define the convolutional layers.
        self.convolutional_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Define the classification layers.
        self.classification_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, number_of_classes)
        )

    def forward(self, input_images):

        # Extract visual features from the input images.
        extracted_features = self.convolutional_layers(input_images)

        # Predict the class scores from the extracted features.
        output_scores = self.classification_layers(extracted_features)

        return output_scores


# Step 4: Create the CNN model.
cnn_model = TrafficSignCNN(number_of_classes=326)

# Step 5: Load the saved weights onto the reconstructed CNN.
saved_cnn_weights = torch.load(
    cnn_model_path,
    map_location="cpu",
    weights_only=True
)

# Strict loading checks whether all parameter names and shapes match.
cnn_model.load_state_dict(saved_cnn_weights, strict=True)

# Step 6: Move the model to the selected device and switch to evaluation mode.
cnn_model = cnn_model.to(device)
cnn_model.eval()

# Step 7: Create a sample input to check the network dimensions.
sample_input = torch.zeros(1, 3, 64, 64).to(device)

with torch.no_grad():
    sample_output = cnn_model(sample_input)

# Step 8: Print the results of the compatibility test.
print("CNN weights loaded successfully.")
print("Input tensor shape:", tuple(sample_input.shape))
print("Output tensor shape:", tuple(sample_output.shape))
print("Number of output classes:", sample_output.shape[1])
print("CNN is ready for the next integration step.")

In [ ]:
# This code selects one real image from the existing CNN validation dataset.
# It resizes the image to 64 by 64 pixels and runs the trained CNN.
# It prints the five classes with the highest prediction scores.
# This code does not train or modify the CNN model.

import os
import json
import random
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

# Step 1: Set the paths of the existing project files.
project_folder = r"."

class_mapping_path = os.path.join(
    project_folder,
    "train",
    "final_class_to_index.json"
)

# Step 2: Find a folder that contains the original CNN classification images.
# We will check common folder names without changing any files.
possible_validation_folders = [
    os.path.join(project_folder, "val"),
    os.path.join(project_folder, "validation"),
    os.path.join(project_folder, "test"),
    os.path.join(project_folder, "CNN_Data_Augmentation_Results", "val")
]

cnn_validation_folder = None

for folder_path in possible_validation_folders:
    if os.path.isdir(folder_path):
        cnn_validation_folder = folder_path
        break

print("CNN validation folder:", cnn_validation_folder)

# Step 3: Load the mapping from class names to output indexes.
with open(class_mapping_path, "r", encoding="utf-8") as mapping_file:
    class_to_index = json.load(mapping_file)

# Create a reverse mapping so that each output index can be converted into a class name.
index_to_class = {
    int(class_index): class_name
    for class_name, class_index in class_to_index.items()
}

# Step 4: Find one image file inside the validation folder.
supported_image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
image_paths = []

if cnn_validation_folder is not None:
    for current_folder, subfolders, filenames in os.walk(cnn_validation_folder):
        for filename in filenames:
            if filename.lower().endswith(supported_image_extensions):
                image_paths.append(os.path.join(current_folder, filename))

if len(image_paths) == 0:
    print("No suitable image was found in the checked validation folders.")
    print("We need to locate the original CNN dataset folders before testing.")
else:
    # Step 5: Select one image so that the test is easy to repeat.
    random.seed(42)
    selected_image_path = random.choice(image_paths)

    print("Selected image:", selected_image_path)

    # Step 6: Load the image and convert it to RGB.
    original_image = Image.open(selected_image_path).convert("RGB")

    # Step 7: Resize the image to 64 by 64 pixels and convert it to a tensor.
    # This is a preliminary transform; the original normalization still needs verification.
    image_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    input_tensor = image_transform(original_image).unsqueeze(0).to(device)

    # Step 8: Run the CNN without calculating gradients.
    with torch.no_grad():
        output_scores = cnn_model(input_tensor)
        probabilities = torch.softmax(output_scores, dim=1)
        top_probabilities, top_indexes = torch.topk(probabilities, k=5, dim=1)

    # Step 9: Display the selected image.
    plt.figure(figsize=(4, 4))
    plt.imshow(original_image)
    plt.title("Selected CNN Test Image")
    plt.axis("off")
    plt.show()

    # Step 10: Print the top five predicted classes.
    print("\nTop 5 CNN predictions:")

    for rank in range(5):
        predicted_index = int(top_indexes[0, rank].item())
        predicted_probability = float(top_probabilities[0, rank].item())
        predicted_class_name = index_to_class.get(
            predicted_index,
            "Unknown class"
        )

        print(
            str(rank + 1) + ".",
            predicted_class_name,
            "| Score:",
            round(predicted_probability, 4)
        )

In [ ]:
# This code checks whether the Jupyter kernel has restarted successfully.
# It only prints a message and does not load or modify any model.

print("Jupyter kernel is working again.")
print("We can continue the project step by step.")

In [ ]:
# This code checks the existing CNN project folders after the Jupyter kernel restart.
# It only lists a small number of directory names and does not load any model.
# It does not train a model or use the GPU.

import os

# Step 1: Set the main folder of our project.
project_folder = r"."

# Step 2: List the main folders inside the project directory.
print("========== MAIN PROJECT FOLDERS ==========")

if os.path.isdir(project_folder):
    main_folder_names = os.listdir(project_folder)

    for folder_name in main_folder_names:
        full_folder_path = os.path.join(project_folder, folder_name)

        if os.path.isdir(full_folder_path):
            print(folder_name)
else:
    print("The project folder could not be found.")

# Step 3: Check the folders where the CNN models were saved.
print("\n========== CNN MODEL FOLDERS ==========")

cnn_folder_paths = [
    os.path.join(project_folder, "Baseline_CNN_Results"),
    os.path.join(project_folder, "CNN_Data_Augmentation_Results"),
    os.path.join(project_folder, "train")
]

for folder_path in cnn_folder_paths:
    print("\nFolder:", folder_path)

    if os.path.isdir(folder_path):
        print("Folder exists.")

        # Print only the first 10 filenames to keep the output short.
        folder_contents = os.listdir(folder_path)

        for filename in folder_contents[:10]:
            print("  ", filename)
    else:
        print("Folder does not exist.")

print("\nFolder inspection finished.")

In [ ]:
# This code checks the organization of the existing CNN training, validation, and test folders.
# It counts the image files and checks whether class mapping files are available.
# It only reads directory information and does not load images or models.

import os

# Step 1: Set the main project directory.
project_folder = r"."

# Step 2: Define the three CNN dataset folders.
dataset_folders = {
    "Training": os.path.join(project_folder, "train"),
    "Validation": os.path.join(project_folder, "val"),
    "Testing": os.path.join(project_folder, "test")
}

# Step 3: Count the image files in each folder.
supported_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for dataset_name, dataset_folder in dataset_folders.items():

    print("\n==========", dataset_name.upper(), "DATASET ==========")
    print("Folder:", dataset_folder)

    if os.path.isdir(dataset_folder):

        image_count = 0

        for filename in os.listdir(dataset_folder):
            if filename.lower().endswith(supported_extensions):
                image_count += 1

        print("Image count:", image_count)

        # Check for class mapping files in the same folder.
        for filename in os.listdir(dataset_folder):
            if (
                "class_to_index" in filename.lower()
                or "mapping" in filename.lower()
            ):
                print("Mapping file found:", filename)

    else:
        print("Folder does not exist.")

# Step 4: Check the main class mapping file used by our CNN.
class_mapping_path = os.path.join(
    project_folder,
    "train",
    "final_class_to_index.json"
)

print("\n========== CLASS MAPPING CHECK ==========")
print("Mapping file exists:", os.path.isfile(class_mapping_path))

print("\nDataset folder inspection finished.")

In [ ]:
# This code searches the project folders for the original CNN training scripts.
# It looks for Python files and Jupyter notebooks with CNN-related names.
# It does not scan image contents, load models, or start training.

import os

# Step 1: Set the main project directory.
project_folder = r"."

# Step 2: Define the folders where training code may be stored.
folders_to_check = [
    project_folder,
    os.path.join(project_folder, "Baseline_CNN_Results"),
    os.path.join(project_folder, "CNN_Data_Augmentation_Results"),
    os.path.join(project_folder, "Project_Backup")
]

# Step 3: Search only for Python scripts and notebook files.
code_file_extensions = (".py", ".ipynb")

print("========== CNN TRAINING CODE SEARCH ==========")

for folder_path in folders_to_check:

    print("\nChecking folder:", folder_path)

    if os.path.isdir(folder_path):

        for filename in os.listdir(folder_path):

            if filename.lower().endswith(code_file_extensions):

                # Print files whose names may relate to CNN training.
                if (
                    "cnn" in filename.lower()
                    or "train" in filename.lower()
                    or "augment" in filename.lower()
                    or filename.lower().endswith(".ipynb")
                ):
                    print(os.path.join(folder_path, filename))

    else:
        print("Folder does not exist.")

print("\nCode file search finished.")

In [ ]:
# This code lists the contents of the project backup folder.
# It checks a limited number of files and folders without opening images or loading models.
# It does not modify the backup or start any training.

import os

# Step 1: Set the path of the project backup folder.
backup_folder = r".\Project_Backup"

# Step 2: Check whether the backup folder exists.
print("========== PROJECT BACKUP INSPECTION ==========")

if os.path.isdir(backup_folder):

    # Step 3: List the main items inside the backup folder.
    backup_items = os.listdir(backup_folder)

    print("Number of items:", len(backup_items))

    for item_name in backup_items:
        item_path = os.path.join(backup_folder, item_name)

        if os.path.isdir(item_path):
            print("[FOLDER]", item_name)

        else:
            print("[FILE]", item_name)

    # Step 4: Check the first level of each subfolder.
    print("\n========== BACKUP SUBFOLDER CONTENTS ==========")

    for item_name in backup_items:

        item_path = os.path.join(backup_folder, item_name)

        if os.path.isdir(item_path):

            print("\nFolder:", item_name)

            try:
                subfolder_items = os.listdir(item_path)

                # Print no more than 15 items from each folder.
                for subfolder_item in subfolder_items[:15]:
                    print("  ", subfolder_item)

            except PermissionError:
                print("  Cannot access this folder.")

else:
    print("The project backup folder does not exist.")

print("\nBackup inspection finished.")

In [ ]:
# This code reads the existing project status and dataset processing reports.
# It also displays the column names and a few rows from the CNN training history files.
# It only reads saved information and does not modify any project files.

import os
import json
import pandas as pd

# Step 1: Set the paths of the project backup files.
project_folder = r"."
backup_folder = os.path.join(project_folder, "Project_Backup")

status_file_path = os.path.join(
    backup_folder,
    "PROJECT_STATUS.txt"
)

processing_report_path = os.path.join(
    backup_folder,
    "Dataset_Manifests",
    "processing_report.json"
)

validation_report_path = os.path.join(
    backup_folder,
    "Dataset_Manifests",
    "validation_processing_report.json"
)

baseline_history_path = os.path.join(
    backup_folder,
    "Baseline_CNN_Results",
    "baseline_cnn_training_history.csv"
)

augmentation_history_path = os.path.join(
    backup_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_training_history.csv"
)

# Step 2: Read and display the project status file.
print("========== PROJECT STATUS ==========")

if os.path.isfile(status_file_path):
    with open(status_file_path, "r", encoding="utf-8") as status_file:
        print(status_file.read())
else:
    print("Project status file was not found.")

# Step 3: Read the dataset processing reports.
report_file_paths = [
    processing_report_path,
    validation_report_path
]

for report_path in report_file_paths:

    print("\n========== REPORT:", os.path.basename(report_path), "==========")

    if os.path.isfile(report_path):
        with open(report_path, "r", encoding="utf-8") as report_file:
            report_data = json.load(report_file)

        print(json.dumps(report_data, indent=4, ensure_ascii=False))
    else:
        print("Report file was not found.")

# Step 4: Read the first few rows of the baseline CNN training history.
print("\n========== BASELINE CNN TRAINING HISTORY ==========")

if os.path.isfile(baseline_history_path):
    baseline_history = pd.read_csv(baseline_history_path)

    print("Columns:", list(baseline_history.columns))
    print(baseline_history.head(5).to_string(index=False))
else:
    print("Baseline training history was not found.")

# Step 5: Read the first few rows of the augmented CNN training history.
print("\n========== AUGMENTED CNN TRAINING HISTORY ==========")

if os.path.isfile(augmentation_history_path):
    augmentation_history = pd.read_csv(augmentation_history_path)

    print("Columns:", list(augmentation_history.columns))
    print(augmentation_history.head(5).to_string(index=False))
else:
    print("Augmented CNN training history was not found.")

print("\nProject information inspection finished.")

In [ ]:
# This code searches the existing project folders for CNN training scripts and notebooks.
# It also checks the saved project metadata for information about image preprocessing.
# The code only reads file names and small text files, so it does not load the CNN model.

import os
import json

project_folder = r"."

print("========== SEARCHING FOR CNN TRAINING CODE ==========")

search_folders = [
    project_folder,
    os.path.join(project_folder, "Project_Backup"),
    os.path.join(project_folder, "train")
]

for folder_path in search_folders:
    print("\nChecking folder:", folder_path)

    if os.path.isdir(folder_path):
        for current_folder, subfolders, file_names in os.walk(folder_path):

            # Skip folders that are unlikely to contain the original training code.
            subfolders[:] = [
                folder_name
                for folder_name in subfolders
                if folder_name not in [
                    ".git",
                    "__pycache__",
                    "images",
                    "labels"
                ]
            ]

            for file_name in file_names:
                lower_file_name = file_name.lower()

                if (
                    lower_file_name.endswith(".py")
                    or lower_file_name.endswith(".ipynb")
                ):
                    if any(
                        keyword in lower_file_name
                        for keyword in [
                            "cnn",
                            "train",
                            "classif",
                            "augment",
                            "model"
                        ]
                    ):
                        print(os.path.join(current_folder, file_name))

print("\n========== CHECKING PREPROCESSING METADATA ==========")

metadata_folder = os.path.join(
    project_folder,
    "Project_Backup",
    "Dataset_Manifests"
)

if os.path.isdir(metadata_folder):
    for file_name in os.listdir(metadata_folder):
        if file_name.lower().endswith(".json"):
            file_path = os.path.join(metadata_folder, file_name)

            print("\nMetadata file:", file_name)

            try:
                with open(file_path, "r", encoding="utf-8") as file:
                    metadata = json.load(file)

                if isinstance(metadata, dict):
                    for key, value in metadata.items():
                        if any(
                            word in key.lower()
                            for word in [
                                "size",
                                "resize",
                                "normal",
                                "transform",
                                "image",
                                "pixel",
                                "channel"
                            ]
                        ):
                            print(key, ":", value)

            except Exception as error:
                print("Could not read this file:", error)

print("\nFinished checking the available training code and metadata.")

In [ ]:
# This code loads the existing augmented CNN model and tests it on one validation image.
# It resizes the image to 64 by 64 pixels and converts the image into an RGB tensor.
# It also loads the saved class mapping and displays the predicted class and confidence.
# This code does not train the model or modify any saved project files.

import os
import json
import random
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

# Step 1: Set the paths to the existing project files.
project_folder = r"."

cnn_model_path = os.path.join(
    project_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_best.pth"
)

class_mapping_path = os.path.join(
    project_folder,
    "train",
    "final_class_to_index.json"
)

validation_folder = os.path.join(
    project_folder,
    "val"
)

# Step 2: Check that the required files and folders exist.
print("========== CHECKING FILE PATHS ==========")

print("CNN model exists:", os.path.isfile(cnn_model_path))
print("Class mapping exists:", os.path.isfile(class_mapping_path))
print("Validation folder exists:", os.path.isdir(validation_folder))

if not os.path.isfile(cnn_model_path):
    raise FileNotFoundError("The augmented CNN model was not found.")

if not os.path.isfile(class_mapping_path):
    raise FileNotFoundError("The CNN class mapping was not found.")

if not os.path.isdir(validation_folder):
    raise FileNotFoundError("The validation image folder was not found.")

# Step 3: Load the class mapping and create the reverse mapping.
with open(class_mapping_path, "r", encoding="utf-8") as mapping_file:
    class_to_index = json.load(mapping_file)

index_to_class = {
    int(class_index): class_name
    for class_name, class_index in class_to_index.items()
}

number_of_classes = len(class_to_index)

print("\nNumber of CNN classes:", number_of_classes)

# Step 4: Define the CNN architecture to match the saved weights.
class TrafficSignCNN(nn.Module):

    def __init__(self, number_of_classes=326):
        super(TrafficSignCNN, self).__init__()

        self.convolutional_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classification_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, number_of_classes)
        )

    def forward(self, input_images):
        extracted_features = self.convolutional_layers(input_images)
        output_scores = self.classification_layers(extracted_features)
        return output_scores

# Step 5: Select the available computing device.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Selected device:", device)

# Step 6: Load the saved CNN weights.
cnn_model = TrafficSignCNN(number_of_classes=number_of_classes)

saved_weights = torch.load(
    cnn_model_path,
    map_location="cpu",
    weights_only=True
)

cnn_model.load_state_dict(saved_weights, strict=True)
cnn_model = cnn_model.to(device)
cnn_model.eval()

print("CNN model loaded successfully.")

# Step 7: Find one image from the validation folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
validation_image_paths = []

for file_name in os.listdir(validation_folder):
    if file_name.lower().endswith(image_extensions):
        full_image_path = os.path.join(validation_folder, file_name)
        validation_image_paths.append(full_image_path)

if len(validation_image_paths) == 0:
    raise FileNotFoundError("No validation images were found.")

# Select one image randomly for this preliminary test.
selected_image_path = random.choice(validation_image_paths)

print("\nSelected validation image:")
print(selected_image_path)

# Step 8: Apply the preliminary image preprocessing.
image_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

with Image.open(selected_image_path) as image_file:
    rgb_image = image_file.convert("RGB")
    input_tensor = image_transform(rgb_image).unsqueeze(0)

input_tensor = input_tensor.to(device)

print("Input tensor shape:", tuple(input_tensor.shape))
print(
    "Input pixel range:",
    float(input_tensor.min().item()),
    "to",
    float(input_tensor.max().item())
)

# Step 9: Run the CNN prediction without calculating gradients.
with torch.no_grad():
    prediction_scores = cnn_model(input_tensor)
    prediction_probabilities = torch.softmax(prediction_scores, dim=1)

    confidence, predicted_index = torch.max(
        prediction_probabilities,
        dim=1
    )

predicted_class_index = int(predicted_index.item())
predicted_class_name = index_to_class[predicted_class_index]
predicted_confidence = float(confidence.item())

# Step 10: Display the preliminary classification result.
print("\n========== CNN TEST RESULT ==========")
print("Predicted class index:", predicted_class_index)
print("Predicted class name:", predicted_class_name)
print("Prediction confidence:", round(predicted_confidence * 100, 2), "%")
print("\nPreliminary CNN test completed.")

In [ ]:
# This code compares CNN predictions with the class labels stored in image filenames.
# It uses the CNN model and class mapping loaded in the previous notebook cell.
# It tests only five images to keep the computation small and safe.

import os
import random
import torch
from PIL import Image
from torchvision import transforms

# Step 1: Find validation images that contain a class label in the filename.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

validation_image_paths = []

for file_name in os.listdir(validation_folder):
    if file_name.lower().endswith(image_extensions):
        full_image_path = os.path.join(validation_folder, file_name)
        validation_image_paths.append(full_image_path)

# Step 2: Randomly select up to five validation images.
number_of_images_to_test = min(5, len(validation_image_paths))

random.seed(42)

selected_image_paths = random.sample(
    validation_image_paths,
    number_of_images_to_test
)

# Step 3: Use the same preliminary preprocessing as the previous test.
image_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

correct_prediction_count = 0

print("========== FIVE-IMAGE CNN CHECK ==========")

# Step 4: Predict the class of each selected image.
for image_number, image_path in enumerate(selected_image_paths, start=1):

    file_name = os.path.basename(image_path)

    # The class label is stored after the final "_object_" section.
    if "_object_" in file_name:
        class_label_part = file_name.split("_object_", 1)[1]

        # Remove the object number before extracting the class name.
        class_label_parts = class_label_part.split("_", 1)

        if len(class_label_parts) > 1:
            true_class_name = class_label_parts[1].rsplit(".", 1)[0]
        else:
            true_class_name = "Unknown"
    else:
        true_class_name = "Unknown"

    # Load the image and convert it to RGB.
    with Image.open(image_path) as image_file:
        rgb_image = image_file.convert("RGB")
        input_tensor = image_transform(rgb_image).unsqueeze(0)

    input_tensor = input_tensor.to(device)

    # Run the prediction without calculating gradients.
    with torch.no_grad():
        prediction_scores = cnn_model(input_tensor)
        prediction_probabilities = torch.softmax(prediction_scores, dim=1)

        confidence, predicted_index = torch.max(
            prediction_probabilities,
            dim=1
        )

    predicted_class_index = int(predicted_index.item())
    predicted_class_name = index_to_class[predicted_class_index]
    predicted_confidence = float(confidence.item())

    # Compare the predicted class with the filename label.
    is_correct = predicted_class_name == true_class_name

    if is_correct:
        correct_prediction_count += 1

    print("\nImage", image_number)
    print("File name:", file_name)
    print("Expected class:", true_class_name)
    print("Predicted class:", predicted_class_name)
    print("Confidence:", round(predicted_confidence * 100, 2), "%")
    print("Correct:", is_correct)

# Step 5: Display the result of this small preliminary check.
print("\n========== SUMMARY ==========")
print("Images tested:", number_of_images_to_test)
print("Correct predictions:", correct_prediction_count)
print(
    "Accuracy on these selected images:",
    round(correct_prediction_count / number_of_images_to_test * 100, 2),
    "%"
)

print("\nThis is a small diagnostic test, not a formal model evaluation.")

In [ ]:
# This code checks whether the saved YOLO and CNN model files exist.
# It also displays their file sizes to confirm that the saved checkpoints are available.
# It does not load, train, or modify either model.

import os

# Step 1: Set the project folder.
project_folder = r"."

# Step 2: Set the path to the saved YOLO detection model.
yolo_model_path = os.path.join(
    project_folder,
    "YOLO_Detection_Results",
    "yolo11n_traffic_sign_retry",
    "weights",
    "best.pt"
)

# Step 3: Set the path to the saved augmented CNN classification model.
cnn_model_path = os.path.join(
    project_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_best.pth"
)

# Step 4: Check both model files.
model_file_paths = [
    ("YOLO Detection Model", yolo_model_path),
    ("CNN Classification Model", cnn_model_path)
]

print("========== MODEL CHECKPOINT VERIFICATION ==========")

for model_name, model_path in model_file_paths:

    print("\nModel:", model_name)
    print("Path:", model_path)

    if os.path.isfile(model_path):

        file_size_in_mb = os.path.getsize(model_path) / (1024 * 1024)

        print("Status: File exists")
        print("File size:", round(file_size_in_mb, 2), "MB")

    else:
        print("Status: File not found")

print("\nModel checkpoint verification finished.")

In [ ]:
# This code loads the existing YOLO detector and the augmented CNN classifier.
# It defines the CNN architecture, loads the saved class mapping, and prepares the image transform.
# It also defines a function that detects traffic signs and classifies each detected crop.
# No model training is performed, and the existing model files are not modified.

import os
import json
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO

# Step 1: Define the paths of the project and the two saved models.
project_folder = r"."

yolo_model_path = os.path.join(
    project_folder,
    "YOLO_Detection_Results",
    "yolo11n_traffic_sign_retry",
    "weights",
    "best.pt"
)

cnn_model_path = os.path.join(
    project_folder,
    "CNN_Data_Augmentation_Results",
    "cnn_augmentation_best.pth"
)

class_mapping_path = os.path.join(
    project_folder,
    "train",
    "final_class_to_index.json"
)

# Step 2: Check whether the required files exist before loading the models.
if not os.path.isfile(yolo_model_path):
    raise FileNotFoundError("The YOLO model file could not be found.")

if not os.path.isfile(cnn_model_path):
    raise FileNotFoundError("The CNN model file could not be found.")

if not os.path.isfile(class_mapping_path):
    raise FileNotFoundError("The CNN class mapping file could not be found.")

print("All required model files were found.")

# Step 3: Select the available computing device.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Selected device:", device)

# Step 4: Load the existing YOLO detection model.
yolo_model = YOLO(yolo_model_path)

print("YOLO detection model loaded successfully.")

# Step 5: Define the CNN architecture that matches the saved CNN weights.
class TrafficSignCNN(nn.Module):

    def __init__(self, number_of_classes=326):
        super(TrafficSignCNN, self).__init__()

        self.convolutional_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classification_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, number_of_classes)
        )

    def forward(self, input_images):
        extracted_features = self.convolutional_layers(input_images)
        output_scores = self.classification_layers(extracted_features)
        return output_scores

# Step 6: Load the mapping between class names and class indices.
with open(class_mapping_path, "r", encoding="utf-8") as mapping_file:
    class_to_index = json.load(mapping_file)

index_to_class = {
    int(class_index): class_name
    for class_name, class_index in class_to_index.items()
}

number_of_classes = len(class_to_index)

print("Number of CNN classes:", number_of_classes)

# Step 7: Create the CNN model and load the saved augmented model weights.
cnn_model = TrafficSignCNN(number_of_classes=number_of_classes)

saved_cnn_weights = torch.load(
    cnn_model_path,
    map_location="cpu",
    weights_only=True
)

cnn_model.load_state_dict(saved_cnn_weights, strict=True)
cnn_model = cnn_model.to(device)
cnn_model.eval()

print("CNN classification model loaded successfully.")

# Step 8: Define the preliminary image preprocessing used in our earlier test.
# Each detected traffic sign crop will be resized to 64 by 64 pixels.
cnn_image_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Step 9: Define a function that detects and classifies traffic signs in an image.
def detect_and_classify_traffic_signs(image_path, detection_confidence=0.25):

    # Check whether the input image exists.
    if not os.path.isfile(image_path):
        raise FileNotFoundError("The input image could not be found.")

    # Open the original image and convert it to RGB.
    with Image.open(image_path) as image_file:
        original_image = image_file.convert("RGB")

    # Run YOLO to find traffic sign bounding boxes.
    detection_results = yolo_model.predict(
        source=image_path,
        conf=detection_confidence,
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )

    # Store the classification results for each detected traffic sign.
    traffic_sign_predictions = []

    # Get the detection result for this single image.
    first_result = detection_results[0]

    # Process each detected bounding box separately.
    if first_result.boxes is not None and len(first_result.boxes) > 0:

        for box_number, detected_box in enumerate(first_result.boxes):

            # Read the bounding box coordinates from YOLO.
            box_coordinates = detected_box.xyxy[0].cpu().tolist()

            x_min = int(box_coordinates[0])
            y_min = int(box_coordinates[1])
            x_max = int(box_coordinates[2])
            y_max = int(box_coordinates[3])

            # Make sure the coordinates stay inside the original image.
            x_min = max(0, min(x_min, original_image.width - 1))
            y_min = max(0, min(y_min, original_image.height - 1))
            x_max = max(x_min + 1, min(x_max, original_image.width))
            y_max = max(y_min + 1, min(y_max, original_image.height))

            # Crop the detected traffic sign from the original image.
            cropped_sign = original_image.crop(
                (x_min, y_min, x_max, y_max)
            )

            # Convert the crop into the input format expected by the CNN.
            cnn_input = cnn_image_transform(cropped_sign).unsqueeze(0)
            cnn_input = cnn_input.to(device)

            # Predict the traffic sign class.
            with torch.no_grad():
                classification_scores = cnn_model(cnn_input)

                classification_probabilities = torch.softmax(
                    classification_scores,
                    dim=1
                )

                class_confidence, predicted_class_index = torch.max(
                    classification_probabilities,
                    dim=1
                )

            # Convert the predicted class index into its class name.
            predicted_index = int(predicted_class_index.item())
            predicted_class_name = index_to_class[predicted_index]
            class_confidence_value = float(class_confidence.item())

            # Read the YOLO detection confidence.
            detection_confidence_value = float(
                detected_box.conf[0].item()
            )

            # Save the detection and classification information.
            prediction_information = {
                "box_number": box_number + 1,
                "bounding_box": (
                    x_min,
                    y_min,
                    x_max,
                    y_max
                ),
                "class_name": predicted_class_name,
                "classification_confidence": class_confidence_value,
                "detection_confidence": detection_confidence_value
            }

            traffic_sign_predictions.append(prediction_information)

    # Return the original image and the prediction information.
    return original_image, traffic_sign_predictions

# Step 10: Display a summary confirming that the integration components are ready.
print("\n========== INTEGRATION SETUP ==========")
print("YOLO model:", yolo_model_path)
print("CNN model:", cnn_model_path)
print("CNN class count:", number_of_classes)
print("CNN input size: 64 x 64")
print("Device:", device)
print("Detection and classification function is ready.")

In [ ]:
# This code selects one validation image and runs the integrated YOLO and CNN pipeline.
# It draws a bounding box around each detected traffic sign.
# It also displays the predicted traffic sign class and both confidence scores.
# The code only performs inference and displays the result without modifying model files.

import os
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Step 1: Set the validation image folder.
validation_folder = r".\val"

# Step 2: Collect image paths from the validation folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

validation_image_paths = []

for file_name in os.listdir(validation_folder):
    if file_name.lower().endswith(image_extensions):
        image_path = os.path.join(validation_folder, file_name)
        validation_image_paths.append(image_path)

if len(validation_image_paths) == 0:
    raise FileNotFoundError("No validation images were found.")

# Step 3: Select one image randomly for the integration test.
random.seed(42)

selected_image_path = random.choice(validation_image_paths)

print("Selected image:")
print(selected_image_path)

# Step 4: Run YOLO detection and CNN classification.
original_image, traffic_sign_predictions = detect_and_classify_traffic_signs(
    selected_image_path,
    detection_confidence=0.25
)

print("\nNumber of detected traffic signs:", len(traffic_sign_predictions))

# Step 5: Create a figure and display the original image.
figure, axis = plt.subplots(1, 1, figsize=(14, 10))

axis.imshow(original_image)
axis.set_title("YOLO Traffic Sign Detection and CNN Classification")
axis.axis("off")

# Step 6: Draw a bounding box and prediction label for each detected sign.
for prediction in traffic_sign_predictions:

    x_min, y_min, x_max, y_max = prediction["bounding_box"]

    predicted_class_name = prediction["class_name"]

    detection_confidence_value = prediction["detection_confidence"]
    classification_confidence_value = prediction["classification_confidence"]

    # Create a rectangle around the detected traffic sign.
    bounding_box_rectangle = patches.Rectangle(
        (x_min, y_min),
        x_max - x_min,
        y_max - y_min,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    axis.add_patch(bounding_box_rectangle)

    # Create a label containing the class and confidence values.
    prediction_label = (
        predicted_class_name
        + "\nYOLO: "
        + str(round(detection_confidence_value * 100, 1))
        + "%"
        + " | CNN: "
        + str(round(classification_confidence_value * 100, 1))
        + "%"
    )

    axis.text(
        x_min,
        max(0, y_min - 5),
        prediction_label,
        color="white",
        fontsize=8,
        verticalalignment="bottom",
        bbox=dict(
            facecolor="green",
            alpha=0.8,
            pad=3
        )
    )

# Step 7: Display the final image.
plt.tight_layout()
plt.show()

# Step 8: Print the detailed predictions in the notebook output.
print("\n========== DETECTION AND CLASSIFICATION DETAILS ==========")

if len(traffic_sign_predictions) == 0:
    print("YOLO did not detect any traffic signs in this image.")
else:
    for prediction in traffic_sign_predictions:
        print("\nDetection number:", prediction["box_number"])
        print("Predicted class:", prediction["class_name"])
        print(
            "YOLO detection confidence:",
            round(prediction["detection_confidence"] * 100, 2),
            "%"
        )
        print(
            "CNN classification confidence:",
            round(prediction["classification_confidence"] * 100, 2),
            "%"
        )

print("\nIntegration test finished.")

In [ ]:
# This code checks whether the Jupyter kernel can run Python and whether CUDA is available.
# It also displays the current GPU memory usage without loading either model again.

import torch

print("Python kernel is working.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_memory = torch.cuda.memory_reserved(0) / (1024 ** 3)

    print("GPU memory allocated:", round(allocated_memory, 2), "GB")
    print("GPU memory reserved:", round(reserved_memory, 2), "GB")

print("GPU status check finished.")

In [ ]:
# This code tests the YOLO detector independently after the Jupyter kernel restart.
# It loads only the YOLO model and runs inference on one validation image.
# It uses a small inference batch and limits the number of detections.
# It does not load the CNN model or modify any saved files.

import os
import random
import torch
from ultralytics import YOLO

# Step 1: Define the path to the existing YOLO model.
project_folder = r"."

yolo_model_path = os.path.join(
    project_folder,
    "YOLO_Detection_Results",
    "yolo11n_traffic_sign_retry",
    "weights",
    "best.pt"
)

# Step 2: Define the validation image folder.
validation_folder = os.path.join(
    project_folder,
    "val"
)

# Step 3: Check the model and image folder.
if not os.path.isfile(yolo_model_path):
    raise FileNotFoundError("The YOLO model file was not found.")

if not os.path.isdir(validation_folder):
    raise FileNotFoundError("The validation image folder was not found.")

# Step 4: Find a few available validation images without loading them into memory.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

validation_image_paths = []

for file_name in os.listdir(validation_folder):
    if file_name.lower().endswith(image_extensions):
        validation_image_paths.append(
            os.path.join(validation_folder, file_name)
        )

if len(validation_image_paths) == 0:
    raise FileNotFoundError("No validation images were found.")

# Step 5: Select one image for the YOLO-only test.
random.seed(42)
selected_image_path = random.choice(validation_image_paths)

print("Selected image:", selected_image_path)
print("Loading YOLO model...")

# Step 6: Load only the YOLO detector.
yolo_model = YOLO(yolo_model_path)

print("YOLO model loaded successfully.")
print("Running one-image inference...")

# Step 7: Run inference with a small, controlled configuration.
with torch.inference_mode():

    detection_results = yolo_model.predict(
        source=selected_image_path,
        imgsz=320,
        batch=1,
        conf=0.25,
        max_det=20,
        device=0,
        verbose=False
    )

# Step 8: Display the number of detected traffic signs.
first_result = detection_results[0]

if first_result.boxes is not None:
    number_of_detections = len(first_result.boxes)
else:
    number_of_detections = 0

print("YOLO inference completed successfully.")
print("Number of detected traffic signs:", number_of_detections)
print("YOLO-only test finished.")

In [ ]:
# This code selects one original road image and tests the YOLO detector.
# It searches the original dataset instead of the cropped CNN validation images.
# It runs inference on only one image and displays the detection results.
# It does not load the CNN model or modify any project files.

import os
import random
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Step 1: Define the original dataset folder and the saved YOLO model.
project_folder = r"."

original_image_folder = os.path.join(
    project_folder,
    "images",
    "val"
)

yolo_model_path = os.path.join(
    project_folder,
    "YOLO_Detection_Results",
    "yolo11n_traffic_sign_retry",
    "weights",
    "best.pt"
)

# Step 2: Check whether the original validation images exist.
if not os.path.isdir(original_image_folder):
    raise FileNotFoundError(
        "The original validation image folder was not found: "
        + original_image_folder
    )

# Step 3: Collect image paths without loading image contents into memory.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
original_image_paths = []

for file_name in os.listdir(original_image_folder):
    if file_name.lower().endswith(image_extensions):
        original_image_paths.append(
            os.path.join(original_image_folder, file_name)
        )

if len(original_image_paths) == 0:
    raise FileNotFoundError("No original validation images were found.")

# Step 4: Select one original image for the test.
random.seed(10)
selected_original_image = random.choice(original_image_paths)

print("Selected original image:")
print(selected_original_image)

# Step 5: Load the YOLO detector.
yolo_model = YOLO(yolo_model_path)

print("\nYOLO model loaded.")
print("Running detection on the original image...")

# Step 6: Run detection with a moderate image size and confidence threshold.
with torch.inference_mode():
    detection_results = yolo_model.predict(
        source=selected_original_image,
        imgsz=640,
        batch=1,
        conf=0.25,
        max_det=50,
        device=0,
        verbose=False
    )

# Step 7: Count the detected traffic signs.
first_result = detection_results[0]

if first_result.boxes is not None:
    number_of_detections = len(first_result.boxes)
else:
    number_of_detections = 0

print("Number of detected traffic signs:", number_of_detections)

# Step 8: Display the original image with YOLO bounding boxes.
annotated_image = first_result.plot()

plt.figure(figsize=(14, 9))
plt.imshow(annotated_image[:, :, ::-1])
plt.title("YOLO Detection on an Original Validation Image")
plt.axis("off")
plt.tight_layout()
plt.show()

print("\nOriginal-image YOLO test finished.")

In [ ]:
# This code checks the actual YOLO dataset folder structure.
# It looks for the training and validation image folders and displays a few image filenames.
# It only reads folder names and file names without loading the images or models.

import os

# Step 1: Set the project and YOLO dataset paths.
project_folder = r"."

yolo_dataset_folder = os.path.join(
    project_folder,
    "YOLO_Detection_Dataset"
)

print("========== CHECKING YOLO DATASET ==========")
print("Dataset folder:", yolo_dataset_folder)
print("Dataset folder exists:", os.path.isdir(yolo_dataset_folder))

# Step 2: Check the folders inside the YOLO dataset.
if os.path.isdir(yolo_dataset_folder):

    print("\nFolders and files in the YOLO dataset:")

    for item_name in os.listdir(yolo_dataset_folder):
        item_path = os.path.join(yolo_dataset_folder, item_name)

        if os.path.isdir(item_path):
            print("[Folder]", item_name)
        else:
            print("[File]", item_name)

    # Step 3: Check common image folder locations.
    possible_image_folders = [
        os.path.join(yolo_dataset_folder, "images", "train"),
        os.path.join(yolo_dataset_folder, "images", "val"),
        os.path.join(yolo_dataset_folder, "images", "valid"),
        os.path.join(yolo_dataset_folder, "train", "images"),
        os.path.join(yolo_dataset_folder, "val", "images"),
        os.path.join(yolo_dataset_folder, "valid", "images")
    ]

    print("\n========== CHECKING IMAGE FOLDERS ==========")

    for image_folder in possible_image_folders:

        print("\nFolder:", image_folder)
        print("Exists:", os.path.isdir(image_folder))

        if os.path.isdir(image_folder):

            image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            image_count = 0
            example_names = []

            for file_name in os.listdir(image_folder):
                if file_name.lower().endswith(image_extensions):
                    image_count += 1

                    if len(example_names) < 3:
                        example_names.append(file_name)

            print("Number of images:", image_count)
            print("Example filenames:", example_names)

else:
    print("The expected YOLO dataset folder was not found.")

print("\nDataset folder inspection finished.")

In [ ]:
# This code tests the existing YOLO detector on one original validation image.
# It uses the confirmed image folder and displays the detection result.
# Only one image is processed, and the CNN model is not loaded.

import os
import random
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Step 1: Set the confirmed paths.
project_folder = r"."

original_image_folder = os.path.join(
    project_folder,
    "YOLO_Detection_Dataset",
    "images",
    "val"
)

yolo_model_path = os.path.join(
    project_folder,
    "YOLO_Detection_Results",
    "yolo11n_traffic_sign_retry",
    "weights",
    "best.pt"
)

# Step 2: Collect the available validation image paths.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
original_image_paths = []

for file_name in os.listdir(original_image_folder):
    if file_name.lower().endswith(image_extensions):
        original_image_paths.append(
            os.path.join(original_image_folder, file_name)
        )

if len(original_image_paths) == 0:
    raise FileNotFoundError("No validation images were found.")

# Step 3: Select one image for the test.
random.seed(10)
selected_original_image = random.choice(original_image_paths)

print("Selected original image:")
print(selected_original_image)

# Step 4: Load the YOLO model.
print("\nLoading YOLO model...")
yolo_model = YOLO(yolo_model_path)
print("YOLO model loaded successfully.")

# Step 5: Run inference on only one image.
print("\nRunning YOLO inference...")

with torch.inference_mode():
    detection_results = yolo_model.predict(
        source=selected_original_image,
        imgsz=640,
        batch=1,
        conf=0.25,
        max_det=50,
        device=0,
        verbose=False
    )

# Step 6: Count the detected traffic signs.
first_result = detection_results[0]

if first_result.boxes is not None:
    number_of_detections = len(first_result.boxes)
else:
    number_of_detections = 0

print("Number of detected traffic signs:", number_of_detections)

# Step 7: Display the image with the predicted bounding boxes.
annotated_image = first_result.plot()

plt.figure(figsize=(14, 9))
plt.imshow(annotated_image[:, :, ::-1])
plt.title("YOLO Detection on an Original Validation Image")
plt.axis("off")
plt.tight_layout()
plt.show()

# Step 8: Display the coordinates and confidence of each detection.
for detection_number, detected_box in enumerate(
    first_result.boxes if first_result.boxes is not None else [],
    start=1
):
    box_coordinates = detected_box.xyxy[0].cpu().tolist()
    detection_score = float(detected_box.conf[0].item())

    print("\nDetection:", detection_number)
    print("Bounding box:", [round(value, 1) for value in box_coordinates])
    print("Detection confidence:", round(detection_score * 100, 2), "%")

print("\nYOLO validation-image test finished.")

In [ ]:
# This code checks whether the Jupyter kernel has recovered after the crash.
# It reports the available GPU and current GPU memory usage.
# It does not load any models or process any images.

import torch

print("========== KERNEL RECOVERY CHECK ==========")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    free_memory, total_memory = torch.cuda.mem_get_info(0)

    print("Total GPU memory:", round(total_memory / (1024 ** 3), 2), "GB")
    print("Currently free GPU memory:", round(free_memory / (1024 ** 3), 2), "GB")

print("Kernel recovery check finished.")